**LIVE RUN, redacted, 2026-11-20**

# Manufacturing KPIs from the live digital-twin pipeline

**The story.** A plant runs a CNC machining cell, an assembly line, a hydraulic press and an
air compressor. Every sensor on that floor streams into Databricks in real time. This notebook
turns that stream into the four numbers a plant manager asks about every morning:

1. **OEE**: how much of the planned capacity actually turned into good running time, per line and cell.
2. **Downtime and alarms**: what stopped us, how often, and which machine and component caused it.
3. **Sensor health**: which machines are drifting toward their limits before they alarm.
4. **Throughput**: is each machine running at its target rate, hour by hour?

**How the data flows** (real-time mode):

```
 sensors -> Kafka -> Spark Real-Time Mode --+--> Lakebase (Postgres) LakeTS hot tier --> app dashboard + SSE live feed
                         |                   |        |
                         |                   |        +--> CDF (Lakehouse Sync) --> Unity Catalog Delta cold tier
                         |                   +--> app alert engine --> alert log (control-plane Lakebase)
                         +--> streaming metrics table (latency, throughput per batch)
```

**Why each tier is queried for what.** The *hot tier* (Lakebase) holds the newest readings for
millisecond reads by the app, so this notebook only asks it for the last few minutes ("live now")
and its freshness. The *cold tier* (Delta in Unity Catalog, kept in sync by CDF) holds the full
history, so every KPI that aggregates hours of data runs there. The *metrics table* describes the
stream itself, and the app's *alert log* holds the alarms the operators saw.

| KPI | Main source | Why |
|---|---|---|
| Pipeline at a glance | Delta + metrics table + Lakebase | freshness at both ends, batch latency |
| OEE | Delta cold tier | millions of readings over the window |
| Downtime and alarms | app alert log (Lakebase) + Delta | alarms are raised by the app, faults are in the telemetry |
| Sensor health | Delta (hourly trends) + Lakebase (live now) | history for drift, hot tier for the current value |
| Throughput | Delta cold tier | hourly rates over the window |

**Run it.** Attach serverless compute and *Run all*. Widgets: `setup_id` (empty = latest
manufacturing pipeline), `mode` (`realtime`, `microbatch`, `zerobus`), `lookback_hours`
(default 24), `redact` (default `true`) and `sample` (synthetic data, runs anywhere). The
notebook only reads (SELECT and metadata calls). Pipeline operations (job health, partitions,
CDF lag) are covered by the
[pipeline health check](../../ops/notebooks/pipeline_health_check.ipynb), not here.
See [the use-case README](../README.md) for permissions and KPI definitions.

In [ ]:
from __future__ import annotations

import os
import re
from dataclasses import dataclass

INDUSTRY = "manufacturing"  # this use case is manufacturing-only
MODES = ("realtime", "microbatch", "zerobus")
IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
_DBUTILS = globals().get("dbutils")  # only defined inside a Databricks notebook
_ENV_PREFIX = "KPI_"
_LOOKBACK_MIN_H, _LOOKBACK_MAX_H = 1, 168
_IDENT_RE = re.compile(r"^[A-Za-z0-9_-]{0,128}$")


@dataclass(frozen=True)
class Params:
    """Validated notebook parameters."""

    setup_id: str
    mode: str
    lookback_hours: int
    redact: bool
    sample: bool


def _declare_widgets() -> None:
    """Create the widgets (no-op outside Databricks)."""
    if _DBUTILS is None:
        return
    w = _DBUTILS.widgets
    w.text("setup_id", "", "Setup id (empty = latest manufacturing setup)")
    w.dropdown("mode", MODES[0], list(MODES), "Pipeline mode")
    w.text("lookback_hours", "24", "Lookback (hours, 1-168)")
    w.dropdown("redact", "true", ["true", "false"], "Redact identifiers")
    w.dropdown("sample", "false", ["true", "false"], "Sample data (no workspace access)")


def _raw_param(name: str, default: str) -> str:
    """Widget value, else ``KPI_<NAME>`` env var, else ``default``."""
    if _DBUTILS is not None:
        try:
            return str(_DBUTILS.widgets.get(name))
        except Exception:  # widget not declared (e.g. job parameters) -> fall through
            pass
    return os.environ.get(_ENV_PREFIX + name.upper(), default)


def _as_bool(text: str, default: bool) -> bool:
    value = text.strip().lower()
    if value in {"1", "true", "yes", "y", "on"}:
        return True
    if value in {"0", "false", "no", "n", "off"}:
        return False
    return default


def load_params() -> tuple[Params, list[str]]:
    """Read and validate every widget; invalid values fall back to safe defaults."""
    notes: list[str] = []
    try:
        lookback = int(float(_raw_param("lookback_hours", "24")))
    except ValueError:
        notes.append("lookback_hours is not a number; using 24")
        lookback = 24
    clamped = max(_LOOKBACK_MIN_H, min(_LOOKBACK_MAX_H, lookback))
    if clamped != lookback:
        notes.append(f"lookback_hours clamped to {clamped}")
    mode = _raw_param("mode", MODES[0]).strip().lower()
    if mode not in MODES:
        notes.append(f"mode={mode!r} is not one of {MODES}; using 'realtime'")
        mode = MODES[0]
    setup_id = _raw_param("setup_id", "").strip()
    if not _IDENT_RE.match(setup_id):
        notes.append("setup_id contains unsupported characters and was ignored")
        setup_id = ""
    # Outside Databricks no workspace is reachable, so default to the synthetic dataset.
    sample_default = "false" if IN_DATABRICKS else "true"
    params = Params(
        setup_id=setup_id,
        mode=mode,
        lookback_hours=clamped,
        redact=_as_bool(_raw_param("redact", "true"), True),
        sample=_as_bool(_raw_param("sample", sample_default), not IN_DATABRICKS),
    )
    return params, notes


def missing_requirements() -> list[str]:
    """pip requirements the live (non-sample) mode needs but the compute lacks."""
    import importlib.metadata
    import importlib.util

    missing: list[str] = []
    try:
        parts = re.findall(r"\d+", importlib.metadata.version("databricks-sdk"))[:3]
        if tuple(int(p) for p in parts) < (0, 133, 0):
            missing.append("databricks-sdk>=0.133.0")
    except importlib.metadata.PackageNotFoundError:
        missing.append("databricks-sdk>=0.133.0")
    if importlib.util.find_spec("pg8000") is None:
        missing.append("pg8000")
    return missing


def ensure_dependencies(params: Params) -> None:
    """On Databricks (live mode only) install missing libraries once and restart Python."""
    if params.sample or not IN_DATABRICKS:
        return
    missing = missing_requirements()
    if not missing:
        return
    shell = globals().get("get_ipython", lambda: None)()
    if shell is None or _DBUTILS is None:
        print("Missing libraries (add them to the Environment): " + ", ".join(missing))
        return
    shell.run_line_magic("pip", "install -q " + " ".join(f"'{m}'" for m in missing))
    print("Installed " + ", ".join(missing) + "; restarting Python (re-run if it stops).")
    _DBUTILS.library.restartPython()


_declare_widgets()
PARAMS, PARAM_NOTES = load_params()
ensure_dependencies(PARAMS)

In [ ]:
from dataclasses import dataclass

# ── Access limits (the notebook is strictly read-only) ────────────────────────
STATEMENT_TIMEOUT_MS = 10_000  # Lakebase control-plane / metadata reads
HOT_STATEMENT_TIMEOUT_MS = 40_000  # bounded hot-tier reads (thousands of partitions to plan)
LAKEBASE_CONNECT_TIMEOUT_S = 10
LAKEBASE_ROW_LIMIT = 2_000
UC_ROW_LIMIT = 5_000
UC_WAIT_TIMEOUT = "50s"
HOT_LIVE_MINUTES = 15  # the hot tier is only asked for the newest minutes ("live now")

# ── Naming conventions mirrored from the app (src/iot_twin) ───────────────────
PG_DATABASE = "databricks_postgres"
APP_BACKEND_PROJECT_PREFIX = "iot-app-backend"
APP_BACKEND_BRANCH = "production"
PRIMARY_ENDPOINT, REPLICA_ENDPOINT = "primary", "replica"
HOT_TABLE = "public.sensor_readings"
HISTORY_TABLE = f"lb__shadow_{INDUSTRY}_sensor_readings_history"  # realtime CDF mirror
METRICS_TABLE = f"{INDUSTRY}_metrics"
# Same row filter the app dashboard and Genie apply to the realtime CDF mirror
# (services/genie/genie_space_content.py telemetry_row_filter).
CDC_ROW_FILTER = f"_pg_change_type IN ('insert', 'update_postimage') AND industry = '{INDUSTRY}'"


# ── Plant model (mirror of src/iot_twin/domain/catalog.py, manufacturing) ─────
@dataclass(frozen=True)
class Asset:
    asset_id: str
    name: str
    role: str  # "cell" (production cell), "line", "machine", "utility"
    component: str
    prefix: str  # sensor-name prefix that belongs to the asset


ASSETS: tuple[Asset, ...] = (
    Asset("mfg-asset-cnc-1", "CNC-1", "cell", "Spindle 3", "cnc_1_"),
    Asset("mfg-asset-assy-1", "Assembly line 1", "line", "Line drive", "assy_line_1_"),
    Asset("mfg-asset-press-1", "Hydraulic press 1", "machine", "Hydraulics", "press_1_"),
    Asset("mfg-asset-compressor-1", "Air compressor 1", "utility", "Air end", "compressor_1_"),
)
FACILITY = "Facility (HVAC)"  # hvac_* sensors: building services, not production equipment
OTHER = "Other"


@dataclass(frozen=True)
class AlertRuleDef:
    """Alert rule as defined in the domain catalog (mirrored to intelligence_alert_rules)."""

    rule_id: str
    sensor_name: str
    operator: str  # "gt" or "lt"
    threshold: float
    severity: str
    fault_code: str


ALERT_RULES: tuple[AlertRuleDef, ...] = (
    AlertRuleDef(
        "mfg-alert-spindle-temp", "cnc_1_spindle_temp", "gt", 60.0, "crit", "M-SPINDLE-OVERTEMP"
    ),
    AlertRuleDef("mfg-alert-vibration", "cnc_1_vibration", "gt", 8.0, "warn", "M-SPINDLE-VIBE"),
    AlertRuleDef("mfg-alert-press", "press_1_pressure", "lt", 120.0, "warn", "M-PRESS-LOW"),
)
RULE_BY_SENSOR = {r.sensor_name: r for r in ALERT_RULES}

# Key sensors for the health-trend section (KPI bindings + alert rules + spindle load).
KEY_SENSORS: tuple[tuple[str, str, str], ...] = (
    ("cnc_1_spindle_temp", "Spindle 3 temperature", "°C"),
    ("cnc_1_vibration", "Spindle 3 vibration", "mm/s"),
    ("cnc_1_spindle_power", "Spindle load (power)", "kW"),
    ("press_1_pressure", "Press hydraulic pressure", "bar"),
)

# Rate sensors used as throughput proxies: (asset, sensor, rate unit, per-hour multiplier, label).
THROUGHPUT_SENSORS: tuple[tuple[str, str, str, float, str], ...] = (
    ("Assembly line 1", "assy_line_1_speed", "m/min", 60.0, "conveyor metres per hour"),
    ("CNC-1", "cnc_1_spindle_speed", "RPM", 60.0, "spindle revolutions per hour"),
)
COUNTER_SENSOR = "press_1_cycle_count"  # would give production counts if it incremented

# Drift flags (sensor-health section): last hour versus the whole window.
DRIFT_PCT_OF_NOMINAL = 5.0  # |mean(last hour) - mean(window)| / nominal, in %
BREACH_PCT_FLAG = 1.0  # share of readings past the alert threshold, in %

In [ ]:
import html as _html
import re
import traceback
from collections.abc import Iterable

_HOST_TLDS = r"(?:com|net|org|io|dev|ai|cloud)"
_REDACTION_RULES: tuple[tuple[str, re.Pattern[str], str], ...] = (
    # Secrets first: these are masked even when redaction is switched off.
    ("dsn", re.compile(r"\b[a-z][a-z0-9+.-]*://[^\s/:@]+:[^\s/@]+@[^\s'\"<>]+", re.I), "<dsn>"),
    ("jwt", re.compile(r"\beyJ[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]*"), "<token>"),
    ("pat", re.compile(r"\b(?:dapi|dkea|dose|dsapi)[a-z0-9]{16,}\b", re.I), "<token>"),
    (
        "kv-secret",
        re.compile(
            r"(?i)\b(password|passwd|pwd|secret|client_secret|token|api[_-]?key|authorization)"
            r"(\s*[=:]\s*)(?:bearer\s+)?[^\s,;'\"<>)}\]]+"
        ),
        r"\1\2<redacted>",
    ),
    ("bearer", re.compile(r"(?i)\bbearer\s+[A-Za-z0-9._~+/=-]{12,}"), "Bearer <token>"),
)
_IDENTIFIER_RULES: tuple[tuple[str, re.Pattern[str], str], ...] = (
    (
        "arn",
        re.compile(r"\barn:aws[a-z-]*:[a-z0-9-]+:[a-z0-9-]*:\d{0,12}:[^\s'\"<>,;)]+", re.I),
        "<arn>",
    ),
    ("email", re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b"), "<email>"),
    (
        "bucket-uri",
        re.compile(r"\b(s3a?|gs|abfss?|wasbs?|r2)://[^\s/'\"<>@]+(?:@[^\s/'\"<>]+)?", re.I),
        r"\1://<bucket>",
    ),
    (
        "workspace-path",
        re.compile(r"(?P<p>/(?:Workspace/)?(?:Users|Repos|Shared/\.[a-z]+))/[^/\s'\"<>]+"),
        r"\g<p>/<user>",
    ),
    ("url-host", re.compile(r"\b(https?)://[^\s/'\"<>]+", re.I), r"\1://<host>"),
    ("lakebase-ep", re.compile(r"(?<![A-Za-z0-9_-])ep-[a-z]+-[a-z]+-[a-z0-9]{4,}"), "<host>"),
    (
        "host",
        re.compile(
            rf"(?<![\w@<>.-])(?:[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?\.){{1,6}}{_HOST_TLDS}\b"
            r"(?::\d{2,5})?",
            re.I,
        ),
        "<host>",
    ),
    (
        "uuid",
        re.compile(r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b", re.I),
        "<uuid>",
    ),
    ("hex-id", re.compile(r"\b[0-9a-f]{32}\b", re.I), "<hex-id>"),
    ("cluster-id", re.compile(r"\b\d{4}-\d{6}-[a-z0-9]{8}\b"), "<cluster-id>"),
    ("long-id", re.compile(r"(?<![\d,.])\d{10,}(?![\d,]|\.\d)"), "<id>"),
)
_MIN_KNOWN_LEN = 3
_TRACE_MAX_CHARS = 1200


class Redactor:
    """Central masking for everything the notebook renders.

    * Secret-looking values (tokens, passwords, DSNs) are *always* masked.
    * With ``enabled`` also hosts, ids, emails, ARNs/accounts, buckets and users are masked,
      plus every exact value registered through :meth:`register` (job ids, catalog names...).
    """

    def __init__(self, enabled: bool = True) -> None:
        self.enabled = enabled
        self._known: dict[str, str] = {}

    def register(self, value: object, label: str) -> None:
        """Mask a runtime-discovered exact value (and, for emails, its user-name part)."""
        text = str(value or "").strip()
        if len(text) < _MIN_KNOWN_LEN:
            return
        self._known[text] = label
        if "@" in text:
            local = text.split("@", 1)[0]
            if len(local) >= 4:
                self._known.setdefault(local, "<user>")

    def register_all(self, values: Iterable[object], label: str) -> None:
        for value in values:
            self.register(value, label)

    def _mask_known(self, text: str) -> str:
        # Longest first so "a.b.c" is masked before its parts.
        for value in sorted(self._known, key=len, reverse=True):
            pattern = rf"(?<![A-Za-z0-9_]){re.escape(value)}(?![A-Za-z0-9_])"
            text = re.sub(pattern, self._known[value], text, flags=re.I)
        return text

    def __call__(self, value: object) -> str:
        text = value if isinstance(value, str) else str(value)
        for _name, pattern, repl in _REDACTION_RULES:
            text = pattern.sub(repl, text)
        if not self.enabled:
            return text
        text = self._mask_known(text)
        for _name, pattern, repl in _IDENTIFIER_RULES:
            text = pattern.sub(repl, text)
        return text

    def exception(self, exc: BaseException, *, with_trace: bool = False) -> str:
        """Redacted, bounded rendering of an exception (including chained causes)."""
        if with_trace:
            raw = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))
        else:
            raw = f"{type(exc).__name__}: {exc}"
            if exc.__cause__ is not None:
                raw += f" <- {type(exc.__cause__).__name__}: {exc.__cause__}"
        cleaned = self(raw)
        if len(cleaned) > _TRACE_MAX_CHARS:
            cleaned = cleaned[:_TRACE_MAX_CHARS] + " ...[truncated]"
        return cleaned

    def html(self, value: object) -> str:
        """Redact then HTML-escape plain text for embedding in markup."""
        return _html.escape(self(value))


REDACT = Redactor(enabled=True)

In [ ]:
import re

_ALLOWED_FIRST_KEYWORDS = frozenset({"select", "with", "describe", "show", "explain"})
_FORBIDDEN_KEYWORDS = re.compile(
    r"\b(insert|update|delete|merge|drop|alter|create|truncate|grant|revoke|copy|call|"
    r"optimize|vacuum|refresh|restore|analyze|reindex|execute|msck)\b",
    re.I,
)
_COMMENT_RE = re.compile(r"--[^\n]*|/\*.*?\*/", re.S)
_STRING_RE = re.compile(r"'(?:[^']|'')*'")
# Server-side functions that would write or change state even inside a SELECT.
_FORBIDDEN_FUNCTIONS = re.compile(
    r"\b(pg_terminate_backend|pg_cancel_backend|pg_reload_conf|pg_switch_wal|pg_create_\w+|"
    r"pg_drop_\w+|set_config|nextval|setval|lo_\w+|dblink\w*|"
    r"lakets\.(?:create|drop|add|remove|enable|disable|run|execute|refresh|reset)\w*)\s*\(",
    re.I,
)


class UnsafeSqlError(ValueError):
    """Raised when a statement is not strictly read-only."""


def assert_read_only(sql: str) -> str:
    """Return ``sql`` unchanged if it is a single read-only statement, else raise.

    Defence in depth on top of a read-only transaction / warehouse permissions:
    one statement, starts with SELECT/WITH/DESCRIBE/SHOW/EXPLAIN, no write/DDL keyword
    outside string literals and comments, no state-changing server functions.
    """
    stripped = _STRING_RE.sub("''", _COMMENT_RE.sub(" ", sql)).strip().rstrip(";").strip()
    if not stripped:
        raise UnsafeSqlError("empty statement")
    if ";" in stripped:
        raise UnsafeSqlError("multiple statements are not allowed")
    first = stripped.split(None, 1)[0].lower()
    if first not in _ALLOWED_FIRST_KEYWORDS:
        raise UnsafeSqlError(f"statement must start with SELECT/WITH/DESCRIBE/SHOW, got {first!r}")
    if match := _FORBIDDEN_KEYWORDS.search(stripped):
        raise UnsafeSqlError(f"forbidden keyword in read-only statement: {match.group(0)!r}")
    if match := _FORBIDDEN_FUNCTIONS.search(stripped):
        raise UnsafeSqlError(f"forbidden function in read-only statement: {match.group(1)!r}")
    return sql

In [ ]:
import math
import re
import sys
from datetime import UTC, datetime
from html import escape
from html.parser import HTMLParser
from typing import Any


def _display_html(markup: str) -> None:
    """Databricks: ``displayHTML``. Jupyter: a bundle with Markdown/plain-text fallbacks."""
    show = globals().get("displayHTML")
    if show is not None:
        show(escape_placeholders(markup))
        return
    from IPython.display import display

    display(text_bundle(markup), raw=True)


def text_bundle(markup: str) -> dict[str, str]:
    """``mime_bundle`` whose text versions are redacted again (defence in depth)."""
    bundle = mime_bundle(markup)
    for key in ("text/markdown", "text/plain"):
        bundle[key] = REDACT(bundle[key])
    return bundle


def emit_html(markup: str) -> None:
    """Render markup after central redaction (the only place HTML leaves the notebook)."""
    _display_html(REDACT(markup))


class _RedactingStream:
    """Wrap stdout/stderr so stray print()/warnings are redacted too."""

    def __init__(self, inner: Any) -> None:
        self._inner = inner

    def write(self, data: str) -> int:
        return self._inner.write(REDACT(data) if isinstance(data, str) else data)

    def __getattr__(self, name: str) -> Any:
        return getattr(self._inner, name)


def install_output_guards() -> None:
    """Redact raw stdout/stderr and uncaught-exception tracebacks (best effort)."""
    for name in ("stdout", "stderr"):
        stream = getattr(sys, name)
        if not isinstance(stream, _RedactingStream):
            setattr(sys, name, _RedactingStream(stream))
    ip = globals().get("get_ipython", lambda: None)()
    if ip is not None and hasattr(ip, "set_custom_exc"):

        def _handler(shell: Any, etype: Any, value: Any, tb: Any, tb_offset: Any = None) -> None:
            value.__traceback__ = tb
            print("Uncaught error (redacted):\n" + REDACT.exception(value, with_trace=True))

        ip.set_custom_exc((Exception,), _handler)


# ── Formatting ────────────────────────────────────────────────────────────────
def num(value: Any) -> float | None:
    """Coerce Decimal/str/int to float; None when impossible or NaN."""
    try:
        out = None if value is None else float(value)
    except (TypeError, ValueError):
        return None
    return None if out is None or math.isnan(out) else out


def fmt_num(value: Any, digits: int = 1) -> str:
    """Number with thousands separators; ``-`` for None/NaN."""
    if isinstance(value, bool):
        return str(value)
    number = num(value)
    if number is None:
        return "-" if value is None or not isinstance(value, str) else value
    if float(number).is_integer():
        return f"{int(number):,}"
    return f"{number:,.{digits}f}"


def fmt_pct(value: Any, digits: int = 1) -> str:
    """Fraction (0-1) as a percentage."""
    number = num(value)
    return "-" if number is None else f"{100 * number:,.{digits}f}%"


def fmt_seconds(value: Any) -> str:
    """Compact duration: 45s, 3.2 min, 5.1 h, 2.4 d."""
    secs = num(value)
    if secs is None:
        return "-"
    if abs(secs) < 90:
        return f"{secs:,.0f}s"
    if abs(secs) < 5400:
        return f"{secs / 60:,.1f} min"
    if abs(secs) < 129_600:
        return f"{secs / 3600:,.1f} h"
    return f"{secs / 86400:,.1f} d"


def fmt_ms(value: Any) -> str:
    ms = num(value)
    if ms is None:
        return "-"
    return f"{ms:,.0f} ms" if ms < 10_000 else f"{ms / 1000:,.1f} s"


def to_dt(value: Any) -> datetime | None:
    """Best-effort conversion of a datetime / ISO string / epoch to aware UTC."""
    if value is None:
        return None
    if isinstance(value, datetime):
        return value if value.tzinfo else value.replace(tzinfo=UTC)
    if isinstance(value, int | float):
        return datetime.fromtimestamp(float(value), UTC)
    text = str(value).strip().replace("Z", "+00:00").replace(" ", "T", 1)
    try:
        parsed = datetime.fromisoformat(text)
    except ValueError:
        return None
    return parsed if parsed.tzinfo else parsed.replace(tzinfo=UTC)


def fmt_ts(value: Any) -> str:
    stamp = to_dt(value)
    return "-" if stamp is None else stamp.astimezone(UTC).strftime("%Y-%m-%d %H:%M UTC")


def fmt_hour(value: Any) -> str:
    stamp = to_dt(value)
    return "-" if stamp is None else stamp.astimezone(UTC).strftime("%m-%d %H:00")


# ── HTML building blocks ──────────────────────────────────────────────────────
_INK = "color:inherit"
_CARD = "border:1px solid rgba(128,128,128,.35);border-radius:8px;padding:8px 12px;margin:6px 0"


class _Raw(str):
    """Marks a cell value as already-safe HTML."""


def raw(markup: str) -> _Raw:
    return _Raw(markup)


def fmt_cell(value: Any) -> str:
    if isinstance(value, datetime):
        return fmt_ts(value)
    return value if isinstance(value, str) else fmt_num(value)


def html_table(
    rows: list[dict[str, Any]], columns: list[str] | None = None, max_rows: int = 40
) -> str:
    """Escaped HTML table from dict rows."""
    if not rows:
        return '<p style="opacity:.7"><i>no rows</i></p>'
    cols = columns or list(rows[0])
    head = "".join(f'<th style="text-align:left;padding:3px 12px">{escape(c)}</th>' for c in cols)
    body = []
    for row in rows[:max_rows]:
        cells = "".join(
            '<td style="padding:3px 12px;font-variant-numeric:tabular-nums">'
            f"{row[c] if isinstance(row.get(c), _Raw) else escape(fmt_cell(row.get(c)))}</td>"
            for c in cols
        )
        body.append(f"<tr>{cells}</tr>")
    more = (
        f"<p><i>{len(rows) - max_rows} more rows not shown</i></p>" if len(rows) > max_rows else ""
    )
    return (
        '<table style="border-collapse:collapse;border-bottom:1px solid rgba(128,128,128,.4)">'
        f'<thead style="border-bottom:1px solid rgba(128,128,128,.6)"><tr>{head}</tr></thead>'
        f"<tbody>{''.join(body)}</tbody></table>{more}"
    )


def kv_table(pairs: list[tuple[str, Any]]) -> str:
    return html_table([{"item": k, "value": v} for k, v in pairs], ["item", "value"], 200)


def heading(text: str, level: int = 3) -> str:
    return f"<h{level} style='margin:12px 0 4px'>{escape(text)}</h{level}>"


def paragraph(text: str) -> str:
    return f"<p style='margin:4px 0;max-width:900px'>{escape(text)}</p>"


_NOTE_COLORS = {"INFO": "#2a78d6", "WARN": "#fab219", "NA": "#898781", "SAMPLE": "#ec835a"}


def note(text: str, kind: str = "INFO") -> str:
    color = _NOTE_COLORS.get(kind, _NOTE_COLORS["INFO"])
    return (
        f'<div style="border-left:4px solid {color};padding:3px 10px;margin:6px 0;max-width:900px">'
        f"{escape(text)}</div>"
    )


def sample_banner() -> str:
    if not PARAMS.sample:
        return ""
    return (
        '<div style="border:2px dashed #ec835a;padding:6px 10px;margin:4px 0;font-weight:700">'
        "SAMPLE DATA: synthetic dataset, no workspace was contacted. Numbers are illustrative."
        "</div>"
    )


def not_available(kpi: str, reason: str) -> str:
    """A clear 'not available from this pipeline' block instead of invented numbers."""
    return (
        f'<div style="{_CARD};border-style:dashed">'
        f"<b>{escape(kpi)}: not available from this pipeline</b>"
        f"<p style='margin:4px 0'>Because {escape(reason)}</p></div>"
    )


def data_source_note(tier: str, tables: list[str], window: str, extra: str = "") -> str:
    """The 'Data source' footer every KPI section ends with."""
    rows = [("tier", tier), ("table", ", ".join(tables) or "-"), ("time window", window)]
    if extra:
        rows.append(("note", extra))
    return heading("Data source", 4) + kv_table(rows)


def stat_tiles(tiles: list[tuple[str, str, str]]) -> str:
    """Row of KPI tiles: (label, value, caption)."""
    cells = "".join(
        f'<div style="{_CARD};min-width:150px;display:inline-block;vertical-align:top;'
        f'margin-right:8px"><div style="font-size:12px;opacity:.75">{escape(label)}</div>'
        f'<div style="font-size:22px;font-weight:600">{escape(value)}</div>'
        f'<div style="font-size:11px;opacity:.7">{escape(caption)}</div></div>'
        for label, value, caption in tiles
    )
    # A hidden table carries the same values for text-only viewers.
    table = html_table(
        [{"KPI": label, "value": value, "detail": caption} for label, value, caption in tiles],
        ["KPI", "value", "detail"],
    )
    return f"<div>{cells}</div><details><summary>table view</summary>{table}</details>"


# ── Text fallbacks (identical copy of ops/html_fallback.py; kept in sync by a test) ──
_BLOCK_TAGS = frozenset({"div", "p", "ul", "ol", "li", "section", "pre", "thead", "tbody"})
_SKIP_TAGS = frozenset({"script", "style"})
_SVG_NS = 'xmlns="http://www.w3.org/2000/svg"'
_PURE_SVG = re.compile(r"^\s*<svg\b[^>]*>.*</svg>\s*$", re.DOTALL)
_SPACES = re.compile(r"[ \t\r\f\v]+")
_BLANKS = re.compile(r"\n{3,}")
# Characters that would otherwise turn text such as ``<host>`` into (hidden) inline HTML.
_MD_SPECIAL = re.compile(r"[\\`*<>]")
# Element names the report can emit; any other bare ``<name>`` is a redaction placeholder.
_HTML_TAGS = frozenset(
    [
        "a",
        "abbr",
        "b",
        "blockquote",
        "br",
        "caption",
        "code",
        "col",
        "colgroup",
        "dd",
        "del",
        "desc",
        "details",
        "div",
        "dl",
        "dt",
        "em",
        "g",
        "h1",
        "h2",
        "h3",
        "h4",
        "h5",
        "h6",
        "hr",
        "i",
        "img",
        "li",
        "line",
        "ol",
        "p",
        "path",
        "polyline",
        "pre",
        "rect",
        "s",
        "small",
        "span",
        "strong",
        "sub",
        "summary",
        "sup",
        "svg",
        "table",
        "tbody",
        "td",
        "text",
        "tfoot",
        "th",
        "thead",
        "title",
        "tr",
        "tspan",
        "u",
        "ul",
    ]
)
_BARE_TAG = re.compile(r"<([a-z][a-z0-9-]*)>")


class _MarkdownBuilder(HTMLParser):
    """Collects text from the report's HTML subset and lays it out as Markdown."""

    def __init__(self, markdown: bool) -> None:
        super().__init__(convert_charrefs=True)
        self._markdown = markdown
        self.out: list[str] = []
        self._skip = 0
        self._heading = 0
        self._table: list[list[str]] | None = None
        self._cell: list[str] | None = None
        self._svg: dict[str, list[str]] | None = None
        self._svg_part = ""

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        if tag in _SKIP_TAGS:
            self._skip += 1
        elif self._svg is not None:
            self._svg_part = tag if tag in ("title", "desc") else ""
        elif tag == "svg":
            self._svg = {"title": [], "desc": []}
        elif tag == "table":
            self._table = []
        elif tag == "tr" and self._table is not None:
            self._table.append([])
        elif tag in ("td", "th") and self._table is not None:
            self._cell = []
        elif re.fullmatch(r"h[1-6]", tag):
            self._heading = int(tag[1])
            self.out.append("\n\n" + "#" * self._heading + " ")
        elif tag == "br":
            self._write(" " if self._cell is not None else "\n")
        elif tag in _BLOCK_TAGS:
            self.out.append("\n")

    def handle_endtag(self, tag: str) -> None:
        if tag in _SKIP_TAGS:
            self._skip = max(0, self._skip - 1)
        elif tag == "svg" and self._svg is not None:
            self._flush_svg()
        elif self._svg is not None:
            self._svg_part = ""
        elif tag in ("td", "th") and self._cell is not None and self._table:
            self._table[-1].append(" ".join("".join(self._cell).split()))
            self._cell = None
        elif tag == "table" and self._table is not None:
            self.out.append("\n\n" + _pipe_table(self._table) + "\n\n")
            self._table = None
        elif re.fullmatch(r"h[1-6]", tag) and self._heading:
            self._heading = 0
            self.out.append("\n\n")
        elif tag in _BLOCK_TAGS:
            self.out.append("\n")

    def handle_data(self, data: str) -> None:
        if self._skip:
            return
        if self._svg is not None:
            if self._svg_part:
                self._svg[self._svg_part].append(data)
            return
        self._write(_MD_SPECIAL.sub(r"\\\g<0>", data) if self._markdown else data)

    def _write(self, text: str) -> None:
        if self._cell is not None:
            self._cell.append(text)
        elif self._table is None:
            self.out.append(text)

    def _flush_svg(self) -> None:
        svg, self._svg = self._svg or {}, None
        title = " ".join("".join(svg.get("title", [])).split()) or "chart"
        desc = " ".join("".join(svg.get("desc", [])).split())
        self._write(f"\n[chart] {title}" + (f": {desc}" if desc else "") + "\n")


def escape_placeholders(markup: str) -> str:
    """Escape redaction placeholders (``<host>``) that would otherwise render as empty tags."""
    return _BARE_TAG.sub(
        lambda m: m.group(0) if m.group(1) in _HTML_TAGS else f"&lt;{m.group(1)}&gt;", markup
    )


def _pipe_table(rows: list[list[str]]) -> str:
    rows = [r for r in rows if r]
    if not rows:
        return ""
    width = max(len(r) for r in rows)
    padded = [[c.replace("|", "\\|") for c in r] + [""] * (width - len(r)) for r in rows]
    lines = ["| " + " | ".join(padded[0]) + " |", "|" + "---|" * width]
    lines += ["| " + " | ".join(r) + " |" for r in padded[1:]]
    return "\n".join(lines)


def _convert(markup: str, markdown: bool) -> str:
    builder = _MarkdownBuilder(markdown)
    builder.feed(markup)
    builder.close()
    lines = [_SPACES.sub(" ", line).strip() for line in "".join(builder.out).split("\n")]
    text = _BLANKS.sub("\n\n", "\n".join(lines)).strip()
    return text or "(empty output)"


def html_to_markdown(markup: str) -> str:
    """Markdown rendering of the report's HTML (text is escaped so nothing is hidden)."""
    return _convert(markup, markdown=True)


def html_to_text(markup: str) -> str:
    """Plain-text rendering: same layout as the Markdown, without escapes."""
    return _convert(markup, markdown=False)


def standalone_svg(markup: str) -> str | None:
    """The markup as a standalone SVG document when it is exactly one ``<svg>`` element."""
    if not _PURE_SVG.match(markup) or markup.count("<svg") != 1:
        return None
    svg = markup.strip()
    return svg if _SVG_NS in svg else svg.replace("<svg", f"<svg {_SVG_NS}", 1)


def mime_bundle(markup: str) -> dict[str, str]:
    """``text/html`` plus Markdown/plain-text fallbacks (and ``image/svg+xml`` for charts)."""
    markup = escape_placeholders(markup)
    bundle = {
        "text/html": markup,
        "text/markdown": html_to_markdown(markup),
        "text/plain": html_to_text(markup),
    }
    svg = standalone_svg(markup)
    if svg is not None:
        bundle["image/svg+xml"] = svg
    return bundle


# ── Dependency-free SVG charts (dataviz reference palette, light + dark) ──────
# Series slots in fixed order (blue, orange); critical red only marks thresholds. Ink and
# gridlines inherit the notebook's text colour (currentColor), so light and dark themes both work.
_CHART_STYLE = (
    '<style type="text/css">'
    ".kv-t{fill:currentColor}.kv-s{fill:currentColor;fill-opacity:.78}"
    ".kv-m{fill:currentColor;fill-opacity:.6}"
    ".kv-g{stroke:currentColor;stroke-opacity:.14}.kv-b{stroke:currentColor;stroke-opacity:.4}"
    ".kv-1{fill:#2a78d6;stroke:#2a78d6}.kv-2{fill:#eb6834;stroke:#eb6834}"
    ".kv-r{stroke:#d03b3b;fill:#d03b3b}.kv-ref{stroke:currentColor;fill:currentColor;"
    "stroke-opacity:.7;fill-opacity:.78}"
    "</style>"
)
_FONT = 'font-family="system-ui,-apple-system,Segoe UI,sans-serif"'


def _svg_open(title: str, desc: str, width: int, height: int) -> str:
    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{escape(title)}" '
        f'width="{width}" height="{height}" viewBox="0 0 {width} {height}" '
        f'style="max-width:100%" {_FONT}>{_CHART_STYLE}'
        f"<title>{escape(title)}</title><desc>{escape(desc)}</desc>"
        f'<text x="0" y="16" font-size="14" font-weight="600" class="kv-t">{escape(title)}</text>'
    )


def _nice_max(value: float) -> float:
    """Round an axis maximum up to 1/2/2.5/5 x 10^k."""
    if value <= 0:
        return 1.0
    exp = 10 ** math.floor(math.log10(value))
    for step in (1, 2, 2.5, 5, 10):
        if value <= step * exp:
            return step * exp
    return 10 * exp


def _round_top_bar(x: float, y: float, w: float, h: float, r: float = 4.0) -> str:
    """Path of a vertical bar anchored to the baseline with a rounded data end (top)."""
    r = max(0.0, min(r, w / 2, h))
    return (
        f"M{x:.1f},{y + h:.1f}V{y + r:.1f}Q{x:.1f},{y:.1f} {x + r:.1f},{y:.1f}"
        f"H{x + w - r:.1f}Q{x + w:.1f},{y:.1f} {x + w:.1f},{y + r:.1f}V{y + h:.1f}Z"
    )


def _round_right_bar(x: float, y: float, w: float, h: float, r: float = 4.0) -> str:
    """Path of a horizontal bar anchored left with a rounded data end (right)."""
    r = max(0.0, min(r, h / 2, w))
    return (
        f"M{x:.1f},{y:.1f}H{x + w - r:.1f}Q{x + w:.1f},{y:.1f} {x + w:.1f},{y + r:.1f}"
        f"V{y + h - r:.1f}Q{x + w:.1f},{y + h:.1f} {x + w - r:.1f},{y + h:.1f}H{x:.1f}Z"
    )


def hbar_chart(
    rows: list[tuple[str, float, str]],
    *,
    title: str,
    x_max: float,
    desc: str,
    reference: tuple[float, str] | None = None,
    width: int = 640,
) -> str:
    """Horizontal bars (one series): (label, value, value text), value labels at bar ends."""
    if not rows:
        return note(f"{title}: no data in the selected window", "NA")
    left, right, top, row_h = 150, 70, 30, 30
    height = top + row_h * len(rows) + 22
    pw = width - left - right
    parts = [_svg_open(title, desc, width, height)]
    for i, (label, value, text) in enumerate(rows):
        y = top + i * row_h + 6
        w = pw * max(0.0, min(value, x_max)) / x_max
        parts.append(
            f'<text x="{left - 8}" y="{y + 13}" font-size="12" text-anchor="end" class="kv-s">'
            f"{escape(label)}</text>"
            f'<path d="{_round_right_bar(left, y, max(w, 1.0), 18)}" class="kv-1"/>'
            f'<text x="{left + w + 6:.1f}" y="{y + 13}" font-size="12" class="kv-t">'
            f"{escape(text)}</text>"
        )
    base_y = top + row_h * len(rows)
    parts.append(
        f'<line x1="{left}" x2="{left}" y1="{top}" y2="{base_y}" class="kv-b" stroke-width="1"/>'
    )
    if reference:
        x = left + pw * reference[0] / x_max
        parts.append(
            f'<line x1="{x:.1f}" x2="{x:.1f}" y1="{top - 4}" y2="{base_y}" class="kv-ref" '
            'stroke-width="1.5" stroke-dasharray="4 3"/>'
            f'<text x="{x:.1f}" y="{base_y + 14}" font-size="11" text-anchor="middle" '
            f'class="kv-s">{escape(reference[1])}</text>'
        )
    parts.append("</svg>")
    return "".join(parts)


def column_chart(
    bars: list[tuple[str, float]],
    *,
    title: str,
    y_label: str,
    desc: str,
    target: tuple[float, str] | None = None,
    width: int = 720,
    height: int = 242,
) -> str:
    """Vertical columns over time (one series) with an optional dashed target line."""
    if not bars:
        return note(f"{title}: no data in the selected window", "NA")
    left, right, top, bottom = 56, 16, 46, 30
    pw, ph = width - left - right, height - top - bottom
    y_hi = _nice_max(max([v for _, v in bars] + ([target[0] * 1.1] if target else [])))
    slot = pw / len(bars)
    gap = 2.0 if slot > 6 else 0.5
    parts = [_svg_open(title, desc, width, height)]
    for i in range(5):
        yv = y_hi * i / 4
        y = top + ph - ph * i / 4
        cls = "kv-b" if i == 0 else "kv-g"
        parts.append(
            f'<line x1="{left}" x2="{width - right}" y1="{y:.1f}" y2="{y:.1f}" class="{cls}" '
            'stroke-width="1"/>'
            f'<text x="{left - 6}" y="{y + 4:.1f}" font-size="10" text-anchor="end" '
            f'class="kv-m">{escape(fmt_num(yv, 0 if y_hi >= 10 else 1))}</text>'
        )
    for idx, (_label, value) in enumerate(bars):
        h = ph * max(value, 0.0) / y_hi
        x = left + idx * slot + gap / 2
        parts.append(
            f'<path d="{_round_top_bar(x, top + ph - h, max(slot - gap, 1.0), h)}" class="kv-1"/>'
        )
    if target:
        y = top + ph - ph * target[0] / y_hi
        parts.append(
            f'<line x1="{left}" x2="{width - right}" y1="{y:.1f}" y2="{y:.1f}" class="kv-ref" '
            'stroke-width="1.5" stroke-dasharray="5 4"/>'
            f'<text x="{width - right}" y="{y - 4:.1f}" font-size="11" text-anchor="end" '
            f'class="kv-s">{escape(target[1])}</text>'
        )
    step = max(1, len(bars) // 6)
    for idx in range(0, len(bars), step):
        x = left + idx * slot + slot / 2
        parts.append(
            f'<text x="{x:.1f}" y="{height - 12}" font-size="10" text-anchor="middle" '
            f'class="kv-m">{escape(bars[idx][0])}</text>'
        )
    parts.append(
        f'<text x="{left}" y="{top - 8}" font-size="11" class="kv-m">{escape(y_label)}</text></svg>'
    )
    return "".join(parts)


def line_chart(
    series: dict[str, list[tuple[float, float, str]]],
    *,
    title: str,
    y_label: str,
    desc: str,
    threshold: tuple[float, str] | None = None,
    width: int = 720,
    height: int = 232,
) -> str:
    """Up to two series over time ((x epoch, y, x label) points) with direct end labels."""
    pts = [p for line in series.values() for p in line]
    if not pts:
        return note(f"{title}: no data in the selected window", "NA")
    left, right, top, bottom = 56, 96, 46, 30
    pw, ph = width - left - right, height - top - bottom
    x_lo, x_hi = min(p[0] for p in pts), max(p[0] for p in pts)
    y_vals = [p[1] for p in pts] + ([threshold[0]] if threshold else [])
    y_lo_raw, y_hi_raw = min(y_vals), max(y_vals)
    pad = (y_hi_raw - y_lo_raw) * 0.1 or abs(y_hi_raw) * 0.1 or 1.0
    y_lo, y_hi = y_lo_raw - pad, y_hi_raw + pad

    def sx(x: float) -> float:
        return left + (pw * (x - x_lo) / (x_hi - x_lo) if x_hi > x_lo else pw / 2)

    def sy(y: float) -> float:
        return top + ph - ph * (y - y_lo) / (y_hi - y_lo)

    parts = [_svg_open(title, desc, width, height)]
    for i in range(5):
        yv = y_lo + (y_hi - y_lo) * i / 4
        cls = "kv-b" if i == 0 else "kv-g"
        parts.append(
            f'<line x1="{left}" x2="{left + pw}" y1="{sy(yv):.1f}" y2="{sy(yv):.1f}" '
            f'class="{cls}" stroke-width="1"/><text x="{left - 6}" y="{sy(yv) + 4:.1f}" '
            f'font-size="10" text-anchor="end" class="kv-m">{escape(fmt_num(yv, 1))}</text>'
        )
    if threshold:
        y = sy(threshold[0])
        parts.append(
            f'<line x1="{left}" x2="{left + pw}" y1="{y:.1f}" y2="{y:.1f}" class="kv-r" '
            'stroke-width="1.5" stroke-dasharray="5 4"/>'
            f'<text x="{left + pw + 4}" y="{y + 4:.1f}" font-size="10" class="kv-r" '
            f'stroke="none">{escape(threshold[1])}</text>'
        )
    legend_x = left
    for idx, (label, line) in enumerate(series.items()):
        cls = f"kv-{idx + 1}"
        ordered = sorted(line)
        if not ordered:
            continue
        coords = " ".join(f"{sx(x):.1f},{sy(y):.1f}" for x, y, _ in ordered)
        parts.append(
            f'<polyline fill="none" class="{cls}" stroke-width="2" stroke-linejoin="round" '
            f'points="{coords}" style="fill:none"/>'
        )
        last_x, last_y, _ = ordered[-1]
        parts.append(
            f'<circle cx="{sx(last_x):.1f}" cy="{sy(last_y):.1f}" r="4" class="{cls}" '
            'stroke="none"/>'
            f'<text x="{sx(last_x) + 8:.1f}" y="{sy(last_y) + 4 - idx * 2:.1f}" font-size="11" '
            f'class="kv-t">{escape(label)}</text>'
        )
        parts.append(
            f'<rect x="{legend_x}" y="{height - 12}" width="10" height="10" rx="2" class="{cls}"/>'
            f'<text x="{legend_x + 14}" y="{height - 3}" font-size="11" class="kv-s">'
            f"{escape(label)}</text>"
        )
        legend_x += 30 + 7 * len(label)
    x_first = sorted(pts)[0][2]
    x_last = sorted(pts)[-1][2]
    parts.append(
        f'<text x="{left}" y="{top + ph + 14}" font-size="10" class="kv-m">{escape(x_first)}</text>'
        f'<text x="{left + pw}" y="{top + ph + 14}" font-size="10" text-anchor="end" '
        f'class="kv-m">{escape(x_last)}</text>'
        f'<text x="{left}" y="{top - 8}" font-size="11" class="kv-m">{escape(y_label)}</text></svg>'
    )
    return "".join(parts)

In [ ]:
from collections.abc import Sequence
from datetime import timedelta

# ── KPI formulas ─────────────────────────────────────────────────────────────
# OEE follows the app's own definition (services/genie/genie_space_content.py), so the
# numbers here match what the app's Genie space answers for the same window:
#   Availability = 1 - fault minutes / observed minutes
#   Performance  = AVG(LEAST(value / nominal_value, 1))   (readings with nominal_value > 0)
#   Quality      = share of readings with min_value <= value <= max_value
#   OEE          = Availability x Performance x Quality   (a missing factor counts as 1)


def safe_ratio(part: float | None, total: float | None) -> float | None:
    """part / total, or None when either side is missing or total is not positive."""
    part, total = num(part), num(total)
    if part is None or total is None or total <= 0:
        return None
    return part / total


def availability(observed_minutes: float | None, fault_minutes: float | None) -> float | None:
    """Share of observed minutes without an active fault (1 - fault / observed)."""
    ratio = safe_ratio(fault_minutes or 0, observed_minutes)
    return None if ratio is None else max(0.0, 1.0 - ratio)


def oee(a: float | None, p: float | None, q: float | None) -> float | None:
    """OEE = A x P x Q. Availability is required; a missing P or Q counts as 1 (app rule)."""
    if a is None:
        return None
    return a * (1.0 if p is None else p) * (1.0 if q is None else q)


def oee_factors(row: dict[str, Any]) -> dict[str, float | None]:
    """A, P, Q and OEE for one aggregated row (observed_min, fault_min, performance, quality)."""
    a = availability(row.get("observed_min"), row.get("fault_min"))
    p, q = num(row.get("performance")), num(row.get("quality"))
    return {"availability": a, "performance": p, "quality": q, "oee": oee(a, p, q)}


def mtba_seconds(count: int | None, first: Any, last: Any) -> float | None:
    """Mean time between alerts: span between the first and last opening / (count - 1).

    Identical to the mean of the gaps between consecutive openings (the sum telescopes).
    """
    start, end = to_dt(first), to_dt(last)
    if not count or count < 2 or start is None or end is None:
        return None
    return max(0.0, (end - start).total_seconds()) / (count - 1)


def mean_gap_seconds(timestamps: Iterable[Any]) -> float | None:
    """Mean gap between consecutive timestamps (reference implementation for MTBA)."""
    stamps = sorted(t for t in (to_dt(v) for v in timestamps) if t is not None)
    if len(stamps) < 2:
        return None
    gaps = [(b - a).total_seconds() for a, b in zip(stamps, stamps[1:], strict=False)]
    return sum(gaps) / len(gaps)


def stopped_seconds(
    events: Sequence[dict[str, Any]], start: datetime, end: datetime, initial_state: str
) -> float:
    """Seconds in [start, end) during which production was 'stopped'.

    ``events`` are production_event rows (state, created_at) inside the window, in any order;
    ``initial_state`` is the state in force at ``start`` (the last event before it).
    """
    state, cursor, total = initial_state, start, 0.0
    for event in sorted(events, key=lambda e: to_dt(e["created_at"]) or start):
        at = min(max(to_dt(event["created_at"]) or start, start), end)
        if state == "stopped":
            total += (at - cursor).total_seconds()
        state, cursor = str(event.get("state") or state), at
    if state == "stopped":
        total += (end - cursor).total_seconds()
    return max(0.0, total)


def drift_pct(
    recent_mean: float | None, window_mean: float | None, nominal: float | None
) -> float | None:
    """(mean of the newest hour - mean of the window) as a % of the nominal value."""
    recent_mean, window_mean, nominal = num(recent_mean), num(window_mean), num(nominal)
    if recent_mean is None or window_mean is None or not nominal:
        return None
    return 100.0 * (recent_mean - window_mean) / abs(nominal)


def percentile(values: Iterable[Any], q: float) -> float | None:
    """Linear-interpolated percentile (q in 0..1) of the non-missing values."""
    data = sorted(v for v in (num(x) for x in values) if v is not None)
    if not data:
        return None
    pos = (len(data) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    return data[lo] + (data[hi] - data[lo]) * (pos - lo)


def weighted_mean(pairs: Iterable[tuple[Any, Any]]) -> float | None:
    """Mean of values weighted by counts: pairs of (value, weight)."""
    total = weight = 0.0
    for value, count in pairs:
        v, c = num(value), num(count)
        if v is None or not c:
            continue
        total += v * c
        weight += c
    return total / weight if weight else None


def counter_production(hourly: Sequence[dict[str, Any]]) -> tuple[bool, float | None]:
    """(is a real counter, units produced) from per-hour first/last counter readings.

    A production counter never decreases within or across hours; anything else (noise around a
    nominal value) is not a count and must not be reported as production.
    """
    rows = sorted(hourly, key=lambda r: to_dt(r["hour"]) or datetime.min.replace(tzinfo=UTC))
    firsts = [num(r.get("first_value")) for r in rows]
    lasts = [num(r.get("last_value")) for r in rows]
    if len(rows) < 2 or None in firsts or None in lasts:
        return False, None
    within = all(b >= a for a, b in zip(firsts, lasts, strict=True))
    across = all(nxt >= prev for prev, nxt in zip(lasts, firsts[1:], strict=False))
    produced = lasts[-1] - firsts[0]
    if not (within and across) or produced <= 0:
        return False, None
    return True, produced


def asset_of(sensor_name: str) -> str:
    """Asset (line / cell / machine) a sensor belongs to, from the plant model."""
    for asset in ASSETS:
        if sensor_name.startswith(asset.prefix):
            return asset.name
    return FACILITY if sensor_name.startswith("hvac_") else OTHER


def asset_of_sensor_id(sensor_id: str | None) -> str:
    """Asset for an alert's catalog sensor id (``mfg-sns-<sensor_name>``)."""
    text = str(sensor_id or "")
    return asset_of(text.split("-sns-", 1)[1]) if "-sns-" in text else OTHER


def component_of(asset_name: str) -> str:
    return next((a.component for a in ASSETS if a.name == asset_name), "-")


def breach(sensor_name: str, value: float | None) -> bool | None:
    """True when ``value`` is past the sensor's alert-rule threshold (None: no rule)."""
    rule, value = RULE_BY_SENSOR.get(sensor_name), num(value)
    if rule is None or value is None:
        return None
    return value > rule.threshold if rule.operator == "gt" else value < rule.threshold


def window_bounds(hours: int, now: datetime | None = None) -> tuple[datetime, datetime]:
    end = now or datetime.now(UTC)
    return end - timedelta(hours=hours), end

In [ ]:
# Every statement is a SELECT; ``assert_read_only`` re-checks each one before it runs.
# Lakebase statements use ``%(name)s`` parameters (pg8000 pyformat); Unity Catalog statements are
# ``str.format`` templates whose identifiers are validated by ``uc_fqn`` before use.
_PG_WINDOW = "now() - make_interval(hours => %(hours)s)"

LAKEBASE_SQL: dict[str, tuple[str, str]] = {
    # name: (target, sql) -- app = control-plane Lakebase, hot = pipeline Lakebase (hot tier)
    "cp_setup_by_id": (
        "app",
        "SELECT id::text AS id, industry, status, warehouse_id, updated_at FROM pipeline_setup"
        " WHERE id::text = %(setup_id)s LIMIT 1",
    ),
    "cp_setup_latest": (
        "app",
        "SELECT id::text AS id, industry, status, warehouse_id, updated_at FROM pipeline_setup"
        " WHERE industry = %(industry)s ORDER BY updated_at DESC LIMIT 1",
    ),
    "cp_deployment": (
        "app",
        "SELECT id::text AS id, mode, status, cdf_enabled, lakebase_branch, delta_table,"
        " created_at, updated_at FROM pipeline_deployment"
        " WHERE setup_id::text = %(setup_id)s AND mode = %(mode)s"
        " ORDER BY updated_at DESC LIMIT 1",
    ),
    "cp_resources": (
        "app",
        "SELECT resource_type, identifier, parent_ref FROM provisioned_resource"
        " WHERE setup_id::text = %(setup_id)s AND resource_type IN"
        " ('uc_table', 'uc_schema', 'lakebase_project', 'lakebase_branch') LIMIT 500",
    ),
    "alerts_by_rule": (
        "app",
        "SELECT a.rule_id, a.sensor_id, a.severity, a.fault_code, count(*) AS opened,"
        " min(e.created_at) AS first_at, max(e.created_at) AS last_at"
        " FROM alert_event e JOIN alert a ON a.id = e.alert_id"
        f" WHERE a.industry = %(industry)s AND e.event_type = 'opened' AND e.created_at >= {_PG_WINDOW}"
        " GROUP BY 1, 2, 3, 4 ORDER BY opened DESC LIMIT 200",
    ),
    "alerts_hourly": (
        "app",
        "SELECT date_trunc('hour', e.created_at) AS hour, count(*) AS opened"
        " FROM alert_event e JOIN alert a ON a.id = e.alert_id"
        f" WHERE a.industry = %(industry)s AND e.event_type = 'opened' AND e.created_at >= {_PG_WINDOW}"
        " GROUP BY 1 ORDER BY 1",
    ),
    "alerts_status": (
        "app",
        "SELECT status, severity, count(*) AS alerts FROM alert WHERE industry = %(industry)s"
        " GROUP BY 1, 2 ORDER BY 1, 2",
    ),
    "production_events": (
        "app",
        "SELECT state, created_at FROM production_event WHERE industry = %(industry)s"
        f" AND created_at >= {_PG_WINDOW} ORDER BY created_at LIMIT 1000",
    ),
    "production_state_before": (
        "app",
        "SELECT state, created_at FROM production_event WHERE industry = %(industry)s"
        f" AND created_at < {_PG_WINDOW} ORDER BY created_at DESC LIMIT 1",
    ),
    # Hot tier: only the newest row and the newest minutes (cheap on thousands of partitions).
    "hot_latest": (
        "hot",
        "SELECT event_time AS max_t, extract(epoch FROM (now() - event_time)) AS lag_s"
        f" FROM {HOT_TABLE} ORDER BY event_time DESC LIMIT 1",
    ),
    "hot_live": (
        "hot",
        "SELECT sensor_name, count(*) AS n, avg(value) AS avg_value, max(value) AS max_value,"
        " bool_or(coalesce(fault_active, false)) AS any_fault"
        f" FROM {HOT_TABLE} WHERE event_time >= now() - make_interval(mins => %(minutes)s)"
        " GROUP BY sensor_name ORDER BY sensor_name",
    ),
    # Row estimate for the window from partition statistics (no scan of the hot table).
    "hot_rows_estimate": (
        "hot",
        "SELECT coalesce(sum(greatest(pc.reltuples, 0)), 0)::bigint AS n_estimate"
        " FROM lakets._chunk_metadata cm JOIN lakets._chronotable_registry c"
        " ON c.id = cm.chronotable_id LEFT JOIN pg_class pc ON pc.oid = to_regclass(cm.chunk_name)"
        " WHERE c.schema_name = 'public' AND c.table_name = 'sensor_readings'"
        f" AND cm.status = 'active' AND cm.range_start <= now() AND cm.range_end > {_PG_WINDOW}",
    ),
}


def _asset_case(column: str = "sensor_name") -> str:
    """SQL CASE mapping a sensor name onto its asset (same rule as ``asset_of``)."""
    whens = " ".join(f"WHEN startswith({column}, '{a.prefix}') THEN '{a.name}'" for a in ASSETS)
    return f"CASE {whens} WHEN startswith({column}, 'hvac_') THEN '{FACILITY}' ELSE '{OTHER}' END"


def _breach_case() -> str:
    """SQL predicate: reading past its alert-rule threshold (same rule as ``breach``)."""
    ops = {"gt": ">", "lt": "<"}
    return " OR ".join(
        f"(sensor_name = '{r.sensor_name}' AND value {ops[r.operator]} {r.threshold})"
        for r in ALERT_RULES
    )


_UC_BASE = (
    "WITH r AS (SELECT sensor_name, event_time, value, nominal_value, min_value, max_value,"
    " coalesce(fault_active, false) AS fault_active, date_trunc('minute', event_time) AS minute,"
    f" {_asset_case()} AS asset FROM {{table}}"
    " WHERE {row_filter}event_time >= current_timestamp() - INTERVAL {hours} HOURS)"
)
_PERFORMANCE = "avg(CASE WHEN nominal_value > 0 THEN least(value / nominal_value, 1) END)"
_QUALITY = (
    "avg(CASE WHEN min_value IS NULL OR max_value IS NULL THEN NULL"
    " WHEN value BETWEEN min_value AND max_value THEN 1.0 ELSE 0.0 END)"
)

UC_SQL: dict[str, str] = {
    "glance": (
        "SELECT count(*) AS n, max(event_time) AS max_t, count(DISTINCT sensor_name) AS sensors,"
        " count_if(event_time >= current_timestamp() - INTERVAL 60 MINUTES) AS n_last_hour,"
        " unix_timestamp(current_timestamp()) - unix_timestamp(max(event_time)) AS lag_s"
        " FROM {table} WHERE {row_filter}event_time >= current_timestamp() - INTERVAL {hours} HOURS"
    ),
    # OEE per sensor, per asset and plant-wide in one pass (GROUPING SETS). Facility (HVAC)
    # sensors are excluded: building services are not production equipment.
    "oee": (
        _UC_BASE + " SELECT asset, sensor_name, grouping_id(asset, sensor_name) AS level,"
        " count(*) AS n, count(DISTINCT minute) AS observed_min,"
        " count(DISTINCT CASE WHEN fault_active THEN minute END) AS fault_min,"
        f" {_PERFORMANCE} AS performance, {_QUALITY} AS quality"
        f" FROM r WHERE asset <> '{FACILITY}'"
        " GROUP BY GROUPING SETS ((asset, sensor_name), (asset), ())"
    ),
    # One row per hour and sensor: feeds sensor trends, drift, throughput and the counter check.
    "hourly": (
        _UC_BASE + " SELECT date_trunc('hour', event_time) AS hour, sensor_name, asset,"
        " count(*) AS n, avg(value) AS avg_value, min(value) AS min_seen, max(value) AS max_seen,"
        " max(nominal_value) AS nominal, max(min_value) AS range_lo, max(max_value) AS range_hi,"
        " count(DISTINCT minute) AS observed_min,"
        " count(DISTINCT CASE WHEN fault_active THEN minute END) AS fault_min,"
        f" count_if({_breach_case()}) AS breaches,"
        " count_if(value < min_value OR value > max_value) AS out_of_range,"
        " min_by(value, event_time) AS first_value, max_by(value, event_time) AS last_value"
        " FROM r GROUP BY 1, 2, 3 ORDER BY 1, 2"
    ),
    "metrics": (
        "SELECT count(*) AS batches, max(timestamp) AS last_batch,"
        " percentile(e2e_p50, 0.5) AS e2e_p50_ms, percentile(e2e_p99, 0.95) AS e2e_p95_of_p99_ms,"
        " avg(input_rps) AS input_rps FROM {table}"
        " WHERE mode = '{mode}' AND timestamp >= current_timestamp() - INTERVAL {hours} HOURS"
    ),
}

In [ ]:
from dataclasses import dataclass, field
from decimal import Decimal

PIPELINES_PROJECT_PREFIX = "iot-pipelines"
_UC_IDENT_RE = re.compile(r"^[A-Za-z0-9_-]+$")


class Unavailable(Exception):
    """A data source could not be used; the message says why (shown to the reader)."""


def uc_fqn(*parts: str) -> str:
    """Validated, backtick-quoted three-part Unity Catalog name (hyphens are legal)."""
    if len(parts) != 3 or not all(_UC_IDENT_RE.match(p or "") for p in parts):
        raise ValueError("unsafe Unity Catalog identifier")
    return ".".join(f"`{p}`" for p in parts)


def classify_error(exc: BaseException) -> str:
    """Short, reader-friendly category for an access failure."""
    text = f"{type(exc).__name__} {exc}".lower()
    if isinstance(exc, Unavailable):
        return str(exc)
    if "timeout" in text or "canceling statement" in text:
        return "TIMEOUT"
    if "permission" in text or "denied" in text or "403" in text or "401" in text:
        return "NO ACCESS"
    if "not found" in text or "does not exist" in text or "404" in text:
        return "NOT FOUND"
    return "ERROR"


def to_plain(value: Any) -> Any:
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, bytes | memoryview):
        return bytes(value).decode("utf-8", "replace")
    return value


def plain_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [{k: to_plain(v) for k, v in row.items()} for row in rows]


@dataclass
class Pipeline:
    """What was resolved about the one manufacturing pipeline in focus."""

    deployed: bool = False
    reason: str = ""
    status: str = "-"
    catalog: str = ""
    schema: str = ""
    cold_table: str = ""
    hot_branch: str = ""
    notes: list[str] = field(default_factory=list)

    @property
    def has_hot_tier(self) -> bool:
        return PARAMS.mode == "realtime"

    @property
    def has_metrics(self) -> bool:
        return PARAMS.mode in ("realtime", "microbatch")  # zerobus is not a Spark stream

    @property
    def row_filter(self) -> str:
        return f"{CDC_ROW_FILTER} AND " if PARAMS.mode == "realtime" else ""

    def fqn(self, table: str | None = None) -> str:
        return uc_fqn(self.catalog, self.schema, table or self.cold_table)

    def display(self, table: str | None = None) -> str:
        """Placeholder-safe table name for the data-source notes (redaction masks the parts)."""
        if not (self.catalog and self.schema):
            return "-"
        return f"{self.catalog}.{self.schema}.{table or self.cold_table}"


class LakebaseSession:
    """One read-only Postgres connection (OAuth token as password, TLS, short timeouts).

    Uses pg8000 (pure Python): loading psycopg2's native library can abort the Python process
    on serverless compute.
    """

    def __init__(self, host: str, user: str, token: str) -> None:
        import ssl

        import pg8000.dbapi

        pg8000.dbapi.paramstyle = "pyformat"  # the queries use %(name)s parameters
        self._conn = pg8000.dbapi.connect(
            user=user,
            host=host,
            port=5432,
            database=PG_DATABASE,
            password=token,
            ssl_context=ssl.create_default_context(),
            timeout=LAKEBASE_CONNECT_TIMEOUT_S + HOT_STATEMENT_TIMEOUT_MS // 1000,
            application_name="manufacturing-kpis",
            startup_params={
                "default_transaction_read_only": "on",
                "statement_timeout": str(STATEMENT_TIMEOUT_MS),
            },
        )

    def query(self, sql: str, params: dict[str, Any], timeout_ms: int) -> list[dict[str, Any]]:
        assert_read_only(sql)
        cur = self._conn.cursor()
        try:
            cur.execute("SET TRANSACTION READ ONLY")
            cur.execute(f"SET LOCAL statement_timeout = {int(timeout_ms)}")
            cur.execute(sql, params)
            names = [d[0] for d in cur.description or []]
            rows = cur.fetchmany(LAKEBASE_ROW_LIMIT)
            return plain_rows([dict(zip(names, row, strict=True)) for row in rows])
        finally:
            cur.close()
            self._conn.rollback()  # never leave a transaction open; nothing was written

    def close(self) -> None:
        try:
            self._conn.close()
        except Exception:
            pass


def _field(obj: Any, *path: str) -> Any:
    for attr in path:
        obj = getattr(obj, attr, None)
        if obj is None:
            return None
    return obj


class LiveBackend:
    """Read-only workspace access: SDK list/get calls, Lakebase SELECTs, UC SELECTs."""

    label = "live workspace"

    def __init__(self) -> None:
        from databricks.sdk import WorkspaceClient

        self.ws = WorkspaceClient()
        self.user = self.ws.current_user.me().user_name or ""
        REDACT.register(self.user, "<user>")
        REDACT.register(str(self.ws.config.host or ""), "<host>")
        self._targets: dict[str, tuple[str, str]] = {}
        self._sessions: dict[str, LakebaseSession] = {}
        self.warehouse_id = ""

    # ── Lakebase ──────────────────────────────────────────────────────────────
    def _projects(self, prefix: str) -> list[str]:
        ids = sorted(p.name.split("/", 1)[1] for p in self.ws.postgres.list_projects() if p.name)
        return [i for i in ids if i.startswith(prefix)]

    def _branch_names(self, project: str) -> set[str]:
        branches = self.ws.postgres.list_branches(parent=f"projects/{project}")
        return {b.name.rsplit("/", 1)[-1] for b in branches if b.name}

    def find_project(self, explicit: str, prefix: str, branch: str) -> str:
        if explicit:
            return explicit
        for project in self._projects(prefix):
            if branch in self._branch_names(project):
                return project
        raise Unavailable(f"NOT FOUND: no Lakebase project {prefix}* with branch {branch!r}")

    def locate(self, target: str, project: str, branch: str, endpoints: tuple[str, ...]) -> None:
        parent = f"projects/{project}/branches/{branch}"
        names = {
            e.name.rsplit("/", 1)[-1]: e.name
            for e in self.ws.postgres.list_endpoints(parent=parent)
        }
        REDACT.register(project, "<lakebase-project>")
        for endpoint_id in endpoints:
            if endpoint_id in names:
                host = _field(
                    self.ws.postgres.get_endpoint(name=names[endpoint_id]),
                    "status",
                    "hosts",
                    "host",
                )
                if host:
                    REDACT.register(str(host), "<host>")
                    self._targets[target] = (names[endpoint_id], str(host))
                    return
        raise Unavailable(f"NOT FOUND: no {endpoints} endpoint on branch {branch!r}")

    def _session(self, target: str) -> LakebaseSession:
        if target not in self._sessions:
            if target not in self._targets:
                raise Unavailable("NOT FOUND: Lakebase target was not resolved")
            endpoint, host = self._targets[target]
            token = self.ws.postgres.generate_database_credential(endpoint=endpoint).token
            self._sessions[target] = LakebaseSession(host, self.user, token)
        return self._sessions[target]

    def lb(self, name: str, **params: Any) -> list[dict[str, Any]]:
        target, sql = LAKEBASE_SQL[name]
        timeout = HOT_STATEMENT_TIMEOUT_MS if target == "hot" else STATEMENT_TIMEOUT_MS
        return self._session(target).query(sql, params, timeout)

    # ── Unity Catalog (notebook Spark first, else the setup's SQL warehouse) ──
    def uc(self, name: str, **fields: Any) -> list[dict[str, Any]]:
        sql = assert_read_only(UC_SQL[name].format(**fields))
        spark = globals().get("spark")
        if spark is not None:
            rows = spark.sql(sql).limit(UC_ROW_LIMIT).collect()
            return plain_rows([r.asDict(recursive=True) for r in rows])
        if not self.warehouse_id:
            raise Unavailable("NO ACCESS: no Spark session and no SQL warehouse recorded")
        return self._warehouse(sql)

    def _warehouse(self, sql: str) -> list[dict[str, Any]]:
        from databricks.sdk.service.sql import ExecuteStatementRequestOnWaitTimeout

        resp = self.ws.statement_execution.execute_statement(
            statement=sql,
            warehouse_id=self.warehouse_id,
            wait_timeout=UC_WAIT_TIMEOUT,
            on_wait_timeout=ExecuteStatementRequestOnWaitTimeout.CANCEL,
            row_limit=UC_ROW_LIMIT,
        )
        state = str(getattr(_field(resp, "status", "state"), "value", ""))
        if state != "SUCCEEDED":
            raise RuntimeError(f"statement {state}: {_field(resp, 'status', 'error', 'message')}")
        cols = _field(resp, "manifest", "schema", "columns") or []
        names = [c.name for c in cols]
        types = [str(getattr(c.type_name, "value", c.type_name)) for c in cols]
        out = []
        for row in _field(resp, "result", "data_array") or []:
            out.append({n: _coerce(v, t) for n, v, t in zip(names, row, types, strict=True)})
        return out

    # ── Pipeline resolution (control plane, read-only) ────────────────────────
    def resolve(self) -> Pipeline:
        pipe = Pipeline()
        project = self.find_project("", APP_BACKEND_PROJECT_PREFIX, APP_BACKEND_BRANCH)
        self.locate("app", project, APP_BACKEND_BRANCH, (PRIMARY_ENDPOINT,))
        if PARAMS.setup_id:
            setups = self.lb("cp_setup_by_id", setup_id=PARAMS.setup_id)
        else:
            setups = self.lb("cp_setup_latest", industry=INDUSTRY)
        if not setups:
            pipe.reason = "no manufacturing pipeline setup exists in the app's control plane"
            return pipe
        setup = setups[0]
        REDACT.register(setup["id"], "<setup-id>")
        REDACT.register(setup.get("warehouse_id"), "<warehouse-id>")
        self.warehouse_id = str(setup.get("warehouse_id") or "")
        deployments = self.lb("cp_deployment", setup_id=setup["id"], mode=PARAMS.mode)
        resources = self.lb("cp_resources", setup_id=setup["id"])
        if not deployments:
            pipe.reason = f"the manufacturing setup has no {PARAMS.mode} deployment"
            return pipe
        dep = deployments[0]
        REDACT.register(dep["id"], "<deployment-id>")
        pipe.status = str(dep.get("status") or "-")
        parts = str(dep.get("delta_table") or "").split(".")
        if len(parts) == 3 and all(parts):
            pipe.catalog, pipe.schema, pipe.cold_table = parts
        else:
            for row in resources:
                ident = str(row.get("identifier") or "").split(".")
                if row.get("resource_type") == "uc_schema" and len(ident) == 2:
                    pipe.catalog, pipe.schema = ident
        if PARAMS.mode == "realtime" or not pipe.cold_table:
            pipe.cold_table = (
                HISTORY_TABLE if PARAMS.mode == "realtime" else f"{INDUSTRY}_{PARAMS.mode}"
            )
        REDACT.register(pipe.catalog, "<catalog>")
        REDACT.register(pipe.schema, "<schema>")
        if not (pipe.catalog and pipe.schema):
            pipe.reason = "the deployment does not record its Unity Catalog location"
            return pipe
        pipe.deployed = pipe.status == "deployed"
        if not pipe.deployed:
            pipe.reason = f"the {PARAMS.mode} deployment status is '{pipe.status}', not 'deployed'"
        if pipe.has_hot_tier:
            pipe.hot_branch = str(dep.get("lakebase_branch") or INDUSTRY)
            ref = next(
                (str(r.get("parent_ref") or "") for r in resources
                 if r.get("resource_type") == "lakebase_branch"),
                "",
            )  # fmt: skip
            explicit = ref.split("/", 1)[1] if ref.startswith("projects/") else ""
            try:
                hot_project = self.find_project(explicit, PIPELINES_PROJECT_PREFIX, pipe.hot_branch)
                self.locate(
                    "hot", hot_project, pipe.hot_branch, (REPLICA_ENDPOINT, PRIMARY_ENDPOINT)
                )
            except Exception as exc:
                pipe.notes.append(f"hot tier not reachable: {classify_error(exc)}")
        return pipe

    def close(self) -> None:
        for session in self._sessions.values():
            session.close()


def _coerce(value: Any, type_name: str) -> Any:
    """Statement Execution returns strings; restore numbers and timestamps."""
    if value is None:
        return None
    if type_name in {"INT", "LONG", "SHORT", "BYTE"}:
        return int(value)
    if type_name in {"DOUBLE", "FLOAT", "DECIMAL"}:
        return float(value)
    if type_name == "TIMESTAMP":
        return to_dt(value)
    if type_name == "BOOLEAN":
        return str(value).lower() == "true"
    return value

In [ ]:
import random
from collections import defaultdict

# (sensor, nominal, lo, hi, relative noise) for the synthetic plant (catalog ranges).
_SAMPLE_SENSORS: tuple[tuple[str, float, float, float, float], ...] = (
    ("cnc_1_spindle_speed", 3500, 0, 8000, 0.04),
    ("cnc_1_spindle_temp", 45, 25, 75, 0.03),
    ("cnc_1_spindle_power", 20, 5, 35, 0.06),
    ("cnc_1_vibration", 2, 0.5, 10, 0.08),
    ("cnc_1_coolant_flow", 25, 10, 40, 0.03),
    ("cnc_1_coolant_temp", 22, 18, 28, 0.02),
    ("assy_line_1_speed", 8, 2, 12, 0.03),
    ("assy_line_1_motor_current", 12, 5, 25, 0.03),
    ("assy_line_1_position", 50, 0, 100, 0.05),
    ("press_1_pressure", 200, 10, 300, 0.05),
    ("press_1_hydraulic_oil_temp", 48, 35, 65, 0.03),
    ("press_1_cycle_count", 123456, 0, 1_000_000, 0.0005),
    ("compressor_1_pressure", 105, 90, 125, 0.02),
    ("compressor_1_motor_temp", 65, 45, 95, 0.02),
    ("compressor_1_flow", 300, 100, 500, 0.04),
    ("hvac_zone_1_temp", 21, 18, 26, 0.02),
    ("hvac_zone_1_humidity", 45, 30, 60, 0.04),
    ("hvac_ahu_1_supply_temp", 15, 12, 18, 0.03),
    ("hvac_ahu_1_return_temp", 22, 19, 25, 0.03),
)
_READINGS_PER_MINUTE = 45  # each synthetic minute stands for this many readings
_FAULT_PERIOD_MIN, _FAULT_LEN_MIN = 47, 11  # spindle over-temperature episodes
_FAULT_EFFECTS = {"cnc_1_spindle_temp": 1.42, "cnc_1_vibration": 2.1, "cnc_1_spindle_power": 0.7}


class SampleBackend:
    """Deterministic synthetic plant: answers the same queries as the live backend."""

    label = "synthetic sample"
    user = ""

    def __init__(self) -> None:
        self.now = datetime.now(UTC).replace(second=0, microsecond=0)
        self.hours = PARAMS.lookback_hours
        self._rng = random.Random(20261120)
        self.readings = self._generate()
        self.warehouse_id = ""

    def _generate(self) -> list[dict[str, Any]]:
        start = self.now - timedelta(hours=self.hours)
        minutes = self.hours * 60
        stop_from, stop_to = minutes - 6 * 60, minutes - 6 * 60 + 25  # one operator stop
        rows = []
        for m in range(minutes):
            stamp = start + timedelta(minutes=m)
            in_fault = m % _FAULT_PERIOD_MIN < _FAULT_LEN_MIN and m > 5
            stopped = stop_from <= m < stop_to
            for name, nominal, lo, hi, noise in _SAMPLE_SENSORS:
                if stopped and name.startswith(("cnc_1_", "assy_line_1_")):
                    continue  # a stopped cell goes quiet (services/production_control.py)
                value = nominal * (1 + self._rng.gauss(0, noise))
                if name == "assy_line_1_speed":
                    value *= 0.93  # the line runs a little below its nominal speed
                if name == "compressor_1_motor_temp" and m > minutes - 180:
                    value *= 1.0 + 0.10 * (m - (minutes - 180)) / 180  # slow upward drift
                fault = False
                if in_fault and name in _FAULT_EFFECTS:
                    value *= _FAULT_EFFECTS[name]
                    fault = True
                if name == "press_1_pressure" and m % 211 == 0:
                    value = 110.0  # an occasional low-pressure stroke
                rows.append(
                    {
                        "sensor_name": name, "event_time": stamp, "value": max(lo, min(hi, value)),
                        "nominal_value": nominal, "min_value": lo, "max_value": hi,
                        "fault_active": fault, "asset": asset_of(name),
                    }
                )  # fmt: skip
        return rows

    # ── query emulation (same column names as the SQL in the queries cell) ──
    def uc(self, name: str, **fields: Any) -> list[dict[str, Any]]:
        return getattr(self, f"_uc_{name}")()

    def lb(self, name: str, **params: Any) -> list[dict[str, Any]]:
        return getattr(self, f"_lb_{name}")(**params)

    @staticmethod
    def _aggregate(rows: list[dict[str, Any]]) -> dict[str, Any]:
        minutes = {r["event_time"] for r in rows}
        faults = {r["event_time"] for r in rows if r["fault_active"]}
        perf = [min(r["value"] / r["nominal_value"], 1.0) for r in rows if r["nominal_value"] > 0]
        qual = [1.0 if r["min_value"] <= r["value"] <= r["max_value"] else 0.0 for r in rows]
        return {
            "n": len(rows) * _READINGS_PER_MINUTE,
            "observed_min": len(minutes),
            "fault_min": len(faults),
            "performance": sum(perf) / len(perf) if perf else None,
            "quality": sum(qual) / len(qual) if qual else None,
        }

    def _uc_glance(self) -> list[dict[str, Any]]:
        latest = max(r["event_time"] for r in self.readings) + timedelta(seconds=52)
        recent = [r for r in self.readings if r["event_time"] >= self.now - timedelta(minutes=60)]
        return [
            {
                "n": len(self.readings) * _READINGS_PER_MINUTE,
                "max_t": latest,
                "sensors": len({r["sensor_name"] for r in self.readings}),
                "n_last_hour": len(recent) * _READINGS_PER_MINUTE,
                "lag_s": 8,
            }
        ]

    def _uc_oee(self) -> list[dict[str, Any]]:
        rows = [r for r in self.readings if r["asset"] != FACILITY]
        groups: dict[tuple, list[dict[str, Any]]] = defaultdict(list)
        for r in rows:
            groups[(r["asset"], r["sensor_name"], 0)].append(r)
            groups[(r["asset"], None, 1)].append(r)
            groups[(None, None, 3)].append(r)
        return [
            {"asset": a, "sensor_name": s, "level": lvl, **self._aggregate(g)}
            for (a, s, lvl), g in groups.items()
        ]

    def _uc_hourly(self) -> list[dict[str, Any]]:
        groups: dict[tuple, list[dict[str, Any]]] = defaultdict(list)
        for r in self.readings:
            hour = r["event_time"].replace(minute=0)
            groups[(hour, r["sensor_name"], r["asset"])].append(r)
        out = []
        for (hour, sensor, asset), g in sorted(groups.items(), key=lambda kv: kv[0][:2]):
            values = [r["value"] for r in g]
            out.append(
                {
                    "hour": hour, "sensor_name": sensor, "asset": asset,
                    "n": len(g) * _READINGS_PER_MINUTE, "avg_value": sum(values) / len(values),
                    "min_seen": min(values), "max_seen": max(values),
                    "nominal": g[0]["nominal_value"], "range_lo": g[0]["min_value"],
                    "range_hi": g[0]["max_value"], "observed_min": len(g),
                    "fault_min": sum(1 for r in g if r["fault_active"]),
                    "breaches": sum(_READINGS_PER_MINUTE for r in g if breach(sensor, r["value"])),
                    "out_of_range": 0, "first_value": values[0], "last_value": values[-1],
                }
            )  # fmt: skip
        return out

    def _uc_metrics(self) -> list[dict[str, Any]]:
        if PARAMS.mode == "zerobus":
            return []
        return [
            {
                "batches": 288, "last_batch": self.now - timedelta(minutes=2),
                "e2e_p50_ms": 180.0 if PARAMS.mode == "realtime" else 4_200.0,
                "e2e_p95_of_p99_ms": 2_400.0 if PARAMS.mode == "realtime" else 21_000.0,
                "input_rps": 14.25,
            }
        ]  # fmt: skip

    def _fault_starts(self) -> list[datetime]:
        start = self.now - timedelta(hours=self.hours)
        period = _FAULT_PERIOD_MIN
        return [start + timedelta(minutes=m) for m in range(period, self.hours * 60, period)]

    def _lb_alerts_by_rule(self, **_: Any) -> list[dict[str, Any]]:
        starts = self._fault_starts()
        vibe = starts[::3]
        press = [self.now - timedelta(hours=self.hours) + timedelta(minutes=m)
                 for m in range(211, self.hours * 60, 211)]  # fmt: skip
        out = []
        for rule, stamps in (
            (ALERT_RULES[0], starts),
            (ALERT_RULES[1], vibe),
            (ALERT_RULES[2], press),
        ):
            if stamps:
                out.append(
                    {
                        "rule_id": rule.rule_id, "sensor_id": f"mfg-sns-{rule.sensor_name}",
                        "severity": rule.severity, "fault_code": rule.fault_code,
                        "opened": len(stamps), "first_at": min(stamps), "last_at": max(stamps),
                    }
                )  # fmt: skip
        return out

    def _lb_alerts_hourly(self, **_: Any) -> list[dict[str, Any]]:
        counts: dict[datetime, int] = defaultdict(int)
        for row in self._lb_alerts_by_rule():
            n, first, last = row["opened"], row["first_at"], row["last_at"]
            step = (last - first) / max(n - 1, 1)
            for i in range(n):
                counts[(first + step * i).replace(minute=0, second=0, microsecond=0)] += 1
        return [{"hour": h, "opened": c} for h, c in sorted(counts.items())]

    def _lb_alerts_status(self, **_: Any) -> list[dict[str, Any]]:
        return [
            {"status": "cleared", "severity": "crit", "alerts": 1},
            {"status": "cleared", "severity": "warn", "alerts": 1},
            {"status": "open", "severity": "warn", "alerts": 1},
        ]

    def _lb_production_events(self, **_: Any) -> list[dict[str, Any]]:
        stop = self.now - timedelta(hours=6)
        return [
            {"state": "stopped", "created_at": stop},
            {"state": "running", "created_at": stop + timedelta(minutes=25)},
        ]

    def _lb_production_state_before(self, **_: Any) -> list[dict[str, Any]]:
        return [{"state": "running", "created_at": self.now - timedelta(days=3)}]

    def _lb_hot_latest(self, **_: Any) -> list[dict[str, Any]]:
        return [{"max_t": self.now + timedelta(seconds=55), "lag_s": 2.4}]

    def _lb_hot_live(self, minutes: int = HOT_LIVE_MINUTES, **_: Any) -> list[dict[str, Any]]:
        since = self.now - timedelta(minutes=minutes)
        groups: dict[str, list[dict[str, Any]]] = defaultdict(list)
        for r in self.readings:
            if r["event_time"] >= since:
                groups[r["sensor_name"]].append(r)
        return [
            {
                "sensor_name": s, "n": len(g) * _READINGS_PER_MINUTE,
                "avg_value": sum(r["value"] for r in g) / len(g),
                "max_value": max(r["value"] for r in g),
                "any_fault": any(r["fault_active"] for r in g),
            }
            for s, g in sorted(groups.items())
        ]  # fmt: skip

    def _lb_hot_rows_estimate(self, **_: Any) -> list[dict[str, Any]]:
        return [{"n_estimate": int(len(self.readings) * _READINGS_PER_MINUTE * 1.01)}]

    def resolve(self) -> Pipeline:
        REDACT.register("sample_catalog", "<catalog>")
        REDACT.register("sample_schema", "<schema>")
        return Pipeline(
            deployed=True, status="deployed", catalog="sample_catalog", schema="sample_schema",
            cold_table=HISTORY_TABLE if PARAMS.mode == "realtime" else f"{INDUSTRY}_{PARAMS.mode}",
            hot_branch=INDUSTRY,
        )  # fmt: skip

    def close(self) -> None:
        return None

## Which pipeline is this?

The notebook finds the manufacturing pipeline in the app's own records (or uses `setup_id`),
then reads only what that pipeline wrote during the lookback window.

In [ ]:
REDACT.enabled = PARAMS.redact
install_output_guards()
RUN_STARTED = datetime.now(UTC)
WINDOW_START, WINDOW_END = window_bounds(PARAMS.lookback_hours, RUN_STARTED)
WINDOW_TEXT = f"last {PARAMS.lookback_hours} h ({fmt_ts(WINDOW_START)} to {fmt_ts(WINDOW_END)})"
HOT_WINDOW_TEXT = f"newest {HOT_LIVE_MINUTES} minutes (bounded read of the hot tier)"


class Data:
    """Cached, source-tagged access to the backend (each query runs at most once)."""

    def __init__(self, backend: Any, pipe: Pipeline) -> None:
        self.backend, self.pipe = backend, pipe
        self._cache: dict[tuple, tuple[str, Any]] = {}
        self.sources: dict[str, str] = {}

    def _run(self, key: tuple, source: str, fn: Any) -> list[dict[str, Any]]:
        if key not in self._cache:
            try:
                if self.backend is None:
                    raise Unavailable(f"NO ACCESS: {self.pipe.reason}")
                self._cache[key] = ("ok", fn())
                self.sources[source] = "reached"
            except Exception as exc:
                self._cache[key] = ("err", exc)
                if self.sources.get(source) != "reached":
                    self.sources[source] = classify_error(exc)
        kind, payload = self._cache[key]
        if kind == "err":
            raise payload
        return payload

    def cold(self, name: str) -> list[dict[str, Any]]:
        """UC Delta cold-tier query over the pipeline's telemetry table and lookback window."""
        if not self.pipe.catalog:
            raise Unavailable(f"NOT DEPLOYED: {self.pipe.reason}")
        fields = {"table": self.pipe.fqn(), "row_filter": self.pipe.row_filter,
                  "hours": PARAMS.lookback_hours}  # fmt: skip
        return self._run(
            ("uc", name), "UC Delta (cold tier)", lambda: self.backend.uc(name, **fields)
        )

    def metrics(self) -> list[dict[str, Any]]:
        if not self.pipe.has_metrics:
            raise Unavailable("NOT APPLICABLE: zerobus writes straight to Delta (no Spark stream)")
        fields = {"table": self.pipe.fqn(METRICS_TABLE), "mode": PARAMS.mode,
                  "hours": PARAMS.lookback_hours}  # fmt: skip
        return self._run(("metrics",), "streaming metrics table",
                         lambda: self.backend.uc("metrics", **fields))  # fmt: skip

    def control(self, name: str) -> list[dict[str, Any]]:
        params = {"industry": INDUSTRY, "hours": PARAMS.lookback_hours}
        return self._run(("cp", name), "control-plane Lakebase (alerts)",
                         lambda: self.backend.lb(name, **params))  # fmt: skip

    def hot(self, name: str) -> list[dict[str, Any]]:
        if not self.pipe.has_hot_tier:
            raise Unavailable(f"NOT APPLICABLE: {PARAMS.mode} has no Lakebase hot tier")
        params = {"hours": PARAMS.lookback_hours, "minutes": HOT_LIVE_MINUTES}
        return self._run(("hot", name), "Lakebase LakeTS (hot tier)",
                         lambda: self.backend.lb(name, **params))  # fmt: skip


def connect() -> tuple[Any, Pipeline]:
    try:
        backend = SampleBackend() if PARAMS.sample else LiveBackend()
    except Exception as exc:
        return None, Pipeline(reason=f"the workspace is not reachable ({classify_error(exc)})")
    try:
        return backend, backend.resolve()
    except Exception as exc:
        reason = f"the app's control plane could not be read ({classify_error(exc)})"
        return backend, Pipeline(reason=reason)


def failure_text(exc: BaseException) -> str:
    """Why a data source could not answer (redacted, one line)."""
    category = classify_error(exc)
    if isinstance(exc, Unavailable):
        return category
    return f"{category}: {REDACT.exception(exc)[:240]}"


def not_deployed_block(section: str) -> str:
    return not_available(
        section,
        f"{PIPE.reason}. Deploy the manufacturing {PARAMS.mode} pipeline from the app's admin "
        "page, or pick another mode with the 'mode' widget.",
    )


def render_header() -> str:
    mode_label = {"realtime": "real-time (Spark Real-Time Mode)", "microbatch": "micro-batch",
                  "zerobus": "Zerobus direct ingest"}[PARAMS.mode]  # fmt: skip
    rows = [
        ("pipeline", f"manufacturing / {mode_label}"),
        ("deployment status", PIPE.status),
        ("telemetry table (cold tier)", PIPE.display() if PIPE.catalog else "-"),
        ("hot tier", "Lakebase LakeTS ChronoTable" if PIPE.has_hot_tier else "none in this mode"),
        ("time window", WINDOW_TEXT),
        ("data", BACKEND.label if BACKEND else "no workspace access"),
        ("redaction", "on" if REDACT.enabled else "off"),
    ]
    blocks = [sample_banner(), heading("Run summary", 3), kv_table(rows)]
    blocks += [note(n, "WARN") for n in PARAM_NOTES + PIPE.notes]
    if not PIPE.deployed:
        blocks.append(not_deployed_block(f"Manufacturing {PARAMS.mode} pipeline"))
    return "".join(blocks)


BACKEND, PIPE = connect()
DATA = Data(BACKEND, PIPE)
SUMMARY: list[dict[str, Any]] = []  # headline numbers, collected by every section


emit_html(render_header())

item,value
pipeline,manufacturing / real-time (Spark Real-Time Mode)
deployment status,deployed
telemetry table (cold tier),<catalog>.<schema>.lb__shadow_manufacturing_sensor_readings_history
hot tier,Lakebase LakeTS ChronoTable
time window,last 24 h (2026-11-19 10:40 UTC to 2026-11-20 10:40 UTC)
data,live workspace
redaction,on


## Pipeline at a glance

Before trusting any KPI, check that the data is current: how old the newest reading is,
how many readings arrive per minute, how long the stream takes end to end, and how much data
each tier holds for the window.

In [ ]:
def freshness_tile(label: str, lag_s: Any, what: str) -> tuple[str, str, str]:
    """Freshness tile; a negative age means the producer clock runs ahead of the database."""
    lag = num(lag_s)
    if lag is not None and lag < 0:
        return (label, "under 1s", f"{what}; stamped {fmt_seconds(-lag)} ahead of the "
                "database clock (producer clock skew)")  # fmt: skip
    return (label, fmt_seconds(lag), what)


def pipeline_at_a_glance() -> None:
    """Freshness, ingest rate, latency and row counts: the minimum context for the KPIs."""
    rows: list[tuple[str, str, str]] = []
    notes: list[str] = []
    try:
        g = DATA.cold("glance")[0]
        n, recent = num(g.get("n")) or 0, num(g.get("n_last_hour")) or 0
        rows.append(
            freshness_tile("Data freshness (Delta)", g.get("lag_s"), "age of the newest reading")
        )
        rows.append(("Ingest rate", f"{fmt_num(recent / 60, 0)} rows/min",
                     f"last 60 min; window average {fmt_num(n / (PARAMS.lookback_hours * 60), 0)}"))  # fmt: skip
        rows.append(("Delta rows in window", fmt_num(n), f"{fmt_num(g.get('sensors'))} sensors"))
        SUMMARY.append({"KPI": "Readings analysed", "value": fmt_num(n), "section": "glance"})
    except Exception as exc:
        notes.append(f"Delta cold tier: {failure_text(exc)}")
    try:
        latest = DATA.hot("hot_latest")
        if latest:
            rows.insert(0, freshness_tile("Data freshness (Lakebase)", latest[0].get("lag_s"),
                                          "age of the newest hot-tier reading"))  # fmt: skip
        est = DATA.hot("hot_rows_estimate")
        if est:
            rows.append(("Lakebase rows in window", f"~{fmt_num(est[0].get('n_estimate'))}",
                         "estimate from partition statistics (no table scan)"))  # fmt: skip
    except Exception as exc:
        notes.append(f"Lakebase hot tier: {failure_text(exc)}")
    try:
        m = DATA.metrics()
        if m and num(m[0].get("batches")):
            rows.append(("End-to-end latency p50", fmt_ms(m[0].get("e2e_p50_ms")),
                         "median of per-batch e2e p50"))  # fmt: skip
            rows.append(("End-to-end latency p95", fmt_ms(m[0].get("e2e_p95_of_p99_ms")),
                         "95th percentile of per-batch e2e p99 (conservative)"))  # fmt: skip
        else:
            notes.append("Streaming metrics: no batches recorded in the window")
    except Exception as exc:
        notes.append(f"Streaming metrics: {failure_text(exc)}")
    blocks = [heading("Pipeline at a glance", 2)]
    if not rows and not PIPE.deployed:
        blocks.append(not_deployed_block("Pipeline at a glance"))
    if rows:
        blocks.append(stat_tiles(rows))
    blocks += [note(n, "NA") for n in notes]
    tables = [PIPE.display()] if PIPE.catalog else []
    if PIPE.has_metrics and PIPE.catalog:
        tables.append(PIPE.display(METRICS_TABLE))
    if PIPE.has_hot_tier:
        tables.append(f"Lakebase {HOT_TABLE} (branch {PIPE.hot_branch or INDUSTRY})")
    tier = "UC Delta cold tier + metrics table" + (
        " + Lakebase hot tier" if PIPE.has_hot_tier else ""
    )
    blocks.append(data_source_note(tier, tables, WINDOW_TEXT))
    emit_html("".join(blocks))


pipeline_at_a_glance()

## Pipeline at a glance

Data freshness (Lakebase)

under 1s

age of the newest hot-tier reading; stamped 11s ahead of the database clock (producer clock skew)

Data freshness (Delta)

22s

age of the newest reading

Ingest rate

17,527 rows/min

last 60 min; window average 17,763

Delta rows in window

25,578,754

19 sensors

Lakebase rows in window

~25,500,146

estimate from partition statistics (no table scan)

End-to-end latency p50

7 ms

median of per-batch e2e p50

End-to-end latency p95

49.4 s

95th percentile of per-batch e2e p99 (conservative)

table view

| KPI | value | detail |
|---|---|---|
| Data freshness (Lakebase) | under 1s | age of the newest hot-tier reading; stamped 11s ahead of the database clock (producer clock skew) |
| Data freshness (Delta) | 22s | age of the newest reading |
| Ingest rate | 17,527 rows/min | last 60 min; window average 17,763 |
| Delta rows in window | 25,578,754 | 19 sensors |
| Lakebase rows in window | ~25,500,146 | estimate from partition statistics (no table scan) |
| End-to-end latency p50 | 7 ms | median of per-batch e2e p50 |
| End-to-end latency p95 | 49.4 s | 95th percentile of per-batch e2e p99 (conservative) |

#### Data source

| item | value |
|---|---|
| tier | UC Delta cold tier + metrics table + Lakebase hot tier |
| table | <catalog>.<schema>.lb__shadow_manufacturing_sensor_readings_history, <catalog>.<schema>.manufacturing_metrics, Lakebase public.sensor_readings (branch manufacturing) |
| time window | last 24 h (2026-11-19 10:40 UTC to 2026-11-20 10:40 UTC) |

## 1. Overall Equipment Effectiveness

OEE multiplies three questions: *was the machine available* (no active fault), *did it run at
its nominal rate* (performance), and *were its readings inside the operating range* (quality).
The definition is the one the app's Genie space uses, so both give the same answer. It is
computed on the Delta cold tier because it aggregates every reading in the window.

In [ ]:
_ROLE = {a.name: a.role for a in ASSETS}


def oee_section() -> None:
    blocks = [heading("1. Overall Equipment Effectiveness (OEE)", 2)]
    try:
        rows = DATA.cold("oee")
    except Exception as exc:
        reason = PIPE.reason if not PIPE.catalog else failure_text(exc)
        emit_html(
            "".join(blocks) + not_available("OEE", f"the telemetry could not be read: {reason}.")
        )
        return
    plant = next((r for r in rows if int(num(r.get("level")) or 0) == 3), None)
    assets = sorted(
        (r for r in rows if int(num(r.get("level")) or 0) == 1),
        key=lambda r: [a.name for a in ASSETS].index(r["asset"]) if r["asset"] in _ROLE else 99,
    )
    sensors = [r for r in rows if int(num(r.get("level")) or 0) == 0]
    if plant is None or not num(plant.get("observed_min")):
        emit_html("".join(blocks) + not_available("OEE", "no production readings in the window."))
        return
    p = oee_factors(plant)
    blocks.append(
        stat_tiles(
            [
                ("Plant OEE", fmt_pct(p["oee"]), "A x P x Q, production assets"),
                ("Availability", fmt_pct(p["availability"]), "minutes without an active fault"),
                ("Performance", fmt_pct(p["performance"]), "rate versus nominal (capped at 100%)"),
                ("Quality", fmt_pct(p["quality"]), "readings inside the operating range"),
            ]
        )
    )
    SUMMARY.append({"KPI": "Plant OEE", "value": fmt_pct(p["oee"]), "section": "1"})
    asset_rows = []
    for r in assets:
        f = oee_factors(r)
        asset_rows.append(
            {
                "asset": r["asset"], "type": _ROLE.get(r["asset"], "-"),
                "availability": fmt_pct(f["availability"]), "performance": fmt_pct(f["performance"]),
                "quality": fmt_pct(f["quality"]), "OEE": fmt_pct(f["oee"]),
                "fault minutes": fmt_num(r.get("fault_min")),
                "observed minutes": fmt_num(r.get("observed_min")),
                "_oee": f["oee"] or 0.0,
            }
        )  # fmt: skip
    blocks.append(heading("OEE by line / cell", 4))
    blocks.append(
        html_table(
            asset_rows,
            [
                "asset",
                "type",
                "availability",
                "performance",
                "quality",
                "OEE",
                "fault minutes",
                "observed minutes",
            ],
        )  # fmt: skip
    )
    worst = min(asset_rows, key=lambda r: r["_oee"]) if asset_rows else None
    if worst:
        SUMMARY.append({"KPI": "Lowest-OEE asset", "value": f"{worst['asset']} ({worst['OEE']})",
                        "section": "1"})  # fmt: skip
    emit_html("".join(blocks))
    bars = [(r["asset"], 100 * r["_oee"], r["OEE"]) for r in asset_rows]
    summary = (
        "; ".join(f"{r['asset']} {r['OEE']}" for r in asset_rows) + f"; plant {fmt_pct(p['oee'])}"
    )
    emit_html(hbar_chart(bars, title="OEE by asset (%)", x_max=100.0, desc=summary))
    sensor_rows = []
    for r in sensors:
        f = oee_factors(r)
        sensor_rows.append(
            {
                "sensor": r["sensor_name"], "asset": r["asset"],
                "availability": fmt_pct(f["availability"]), "performance": fmt_pct(f["performance"]),
                "quality": fmt_pct(f["quality"]), "OEE": fmt_pct(f["oee"]), "_oee": f["oee"] or 0.0,
            }
        )  # fmt: skip
    sensor_rows.sort(key=lambda r: r["_oee"])
    emit_html(
        heading("Per sensor (lowest OEE first)", 4)
        + html_table(
            sensor_rows, ["sensor", "asset", "availability", "performance", "quality", "OEE"], 25
        )
        + note(
            "Formula (the app's own definition, so Genie answers the same numbers): "
            "Availability = 1 - fault minutes / observed minutes; Performance = "
            "AVG(LEAST(value / nominal_value, 1)); Quality = share of readings with "
            "min_value <= value <= max_value; OEE = A x P x Q. Minutes are distinct minutes "
            "with a reading; at asset and plant level a minute is a fault minute when any of "
            "the asset's sensors reports fault_active. This is an approximation from sensor "
            "telemetry, not from production counts (the pipeline carries none, see section 4). "
            "Facility (HVAC) sensors are excluded."
        )
        + data_source_note(
            "UC Delta cold tier (Lakebase -> CDF -> Delta)"
            if PARAMS.mode == "realtime"
            else "UC Delta table",
            [PIPE.display()],
            WINDOW_TEXT,
            "Heavy aggregation runs on Delta, not the hot tier: the full window is millions "
            "of readings and Delta scans them cheaply, while the hot tier serves the newest rows.",
        )  # fmt: skip
    )


oee_section()

## 1. Overall Equipment Effectiveness (OEE)

Plant OEE

58.7%

A x P x Q, production assets

Availability

60.0%

minutes without an active fault

Performance

97.8%

rate versus nominal (capped at 100%)

Quality

100.0%

readings inside the operating range

table view

| KPI | value | detail |
|---|---|---|
| Plant OEE | 58.7% | A x P x Q, production assets |
| Availability | 60.0% | minutes without an active fault |
| Performance | 97.8% | rate versus nominal (capped at 100%) |
| Quality | 100.0% | readings inside the operating range |

#### OEE by line / cell

| asset | type | availability | performance | quality | OEE | fault minutes | observed minutes |
|---|---|---|---|---|---|---|---|
| CNC-1 | cell | 60.0% | 97.6% | 100.0% | 58.6% | 576 | 1,441 |
| Assembly line 1 | line | 100.0% | 97.6% | 100.0% | 97.6% | 0 | 1,441 |
| Hydraulic press 1 | machine | 100.0% | 98.4% | 100.0% | 98.4% | 0 | 1,440 |
| Air compressor 1 | utility | 100.0% | 97.7% | 100.0% | 97.7% | 0 | 1,441 |

sensor,asset,availability,performance,quality,OEE
cnc_1_spindle_temp,CNC-1,60.0%,98.9%,100.0%,59.4%
cnc_1_vibration,CNC-1,100.0%,95.3%,100.0%,95.3%
cnc_1_spindle_power,CNC-1,100.0%,96.5%,100.0%,96.5%
compressor_1_pressure,Air compressor 1,100.0%,96.9%,100.0%,96.9%
assy_line_1_position,Assembly line 1,100.0%,97.0%,100.0%,97.0%
cnc_1_spindle_speed,CNC-1,100.0%,97.0%,100.0%,97.0%
assy_line_1_speed,Assembly line 1,100.0%,97.2%,100.0%,97.2%
press_1_pressure,Hydraulic press 1,100.0%,97.3%,100.0%,97.3%
compressor_1_flow,Air compressor 1,100.0%,97.9%,100.0%,97.9%
compressor_1_motor_temp,Air compressor 1,100.0%,98.2%,100.0%,98.2%


## 2. Downtime and alarms

Alarms come from the app's alert engine, which evaluates the live feed against the alert rules
of the plant model and records every alert episode in its Lakebase alert log. Fault minutes come
from the telemetry itself (the `fault_active` flag), and operator stops from the app's
production-control log.

In [ ]:
def _combine_groups(groups: list[dict[str, Any]]) -> dict[str, Any]:
    """Merge alert groups: total openings, earliest first, latest last (MTBA input)."""
    firsts = [to_dt(g["first_at"]) for g in groups if g.get("first_at")]
    lasts = [to_dt(g["last_at"]) for g in groups if g.get("last_at")]
    return {
        "opened": sum(int(num(g.get("opened")) or 0) for g in groups),
        "first_at": min(firsts) if firsts else None,
        "last_at": max(lasts) if lasts else None,
    }


def _fault_minutes_by_asset() -> dict[str, float]:
    try:
        rows = DATA.cold("oee")
    except Exception:
        return {}
    return {
        r["asset"]: num(r.get("fault_min")) or 0.0
        for r in rows
        if int(num(r.get("level")) or 0) == 1
    }


def _operator_stop_seconds() -> float | None:
    events = DATA.control("production_events")
    before = DATA.control("production_state_before")
    initial = str(before[0]["state"]) if before else "running"
    return stopped_seconds(events, WINDOW_START, WINDOW_END, initial)


def alarms_section() -> None:
    blocks = [heading("2. Downtime and alarms", 2)]
    try:
        groups = DATA.control("alerts_by_rule")
    except Exception as exc:
        emit_html(
            "".join(blocks)
            + not_available(
                "Alert and fault counts",
                f"the app's alert log could not be read: {failure_text(exc)}.",
            )
        )
        groups = None
    fault_min = _fault_minutes_by_asset()
    try:
        stop_s = _operator_stop_seconds()
    except Exception:
        stop_s = None
    if groups is None:
        if fault_min:
            emit_html(
                heading("Fault downtime from telemetry", 4)
                + html_table(
                    [{"asset": a, "fault minutes": fmt_num(m)} for a, m in fault_min.items()]
                )
            )
        return
    total = _combine_groups(groups)
    crit = sum(int(num(g.get("opened")) or 0) for g in groups if g.get("severity") == "crit")
    plant_mtba = mtba_seconds(total["opened"], total["first_at"], total["last_at"])
    tiles = [
        ("Alerts opened", fmt_num(total["opened"]), f"in the {PARAMS.lookback_hours} h window"),
        ("Critical alerts", fmt_num(crit), "severity crit"),
        ("Mean time between alerts", fmt_seconds(plant_mtba), "plant-wide"),
        ("Fault downtime", f"{fmt_num(sum(fault_min.values()))} min",
         "sum of asset fault minutes (telemetry)"),
        ("Operator stops", "-" if stop_s is None else fmt_seconds(stop_s),
         "production cell stopped from the app"),
    ]  # fmt: skip
    blocks.append(stat_tiles(tiles))
    SUMMARY.append({"KPI": "Alerts opened", "value": fmt_num(total["opened"]), "section": "2"})
    SUMMARY.append(
        {"KPI": "Mean time between alerts", "value": fmt_seconds(plant_mtba), "section": "2"}
    )
    by_asset: dict[str, list[dict[str, Any]]] = {}
    for g in groups:
        by_asset.setdefault(asset_of_sensor_id(g.get("sensor_id")), []).append(g)
    asset_rows = []
    for asset, items in by_asset.items():
        merged = _combine_groups(items)
        asset_rows.append(
            {
                "asset": asset, "component": component_of(asset), "alerts": merged["opened"],
                "critical": sum(int(num(i.get("opened")) or 0) for i in items if i.get("severity") == "crit"),
                "MTBA": fmt_seconds(mtba_seconds(merged["opened"], merged["first_at"], merged["last_at"])),
                "fault minutes (telemetry)": fmt_num(fault_min.get(asset)),
            }
        )  # fmt: skip
    asset_rows.sort(key=lambda r: -r["alerts"])
    if asset_rows:
        SUMMARY.append({"KPI": "Top offending asset", "value":
                        f"{asset_rows[0]['asset']} ({fmt_num(asset_rows[0]['alerts'])} alerts)",
                        "section": "2"})  # fmt: skip
    blocks.append(heading("Top offending assets and components", 4))
    blocks.append(html_table(asset_rows))
    code_rows = [
        {
            "fault code": g.get("fault_code") or "-", "rule": g.get("rule_id") or "-",
            "severity": g.get("severity") or "-", "alerts": int(num(g.get("opened")) or 0),
            "first": fmt_ts(g.get("first_at")), "last": fmt_ts(g.get("last_at")),
            "MTBA": fmt_seconds(mtba_seconds(num(g.get("opened")), g.get("first_at"), g.get("last_at"))),
        }
        for g in groups
    ]  # fmt: skip
    blocks.append(heading("By fault code", 4) + html_table(code_rows, max_rows=15))
    emit_html("".join(blocks))
    try:
        hourly = DATA.control("alerts_hourly")
    except Exception as exc:
        hourly = []
        emit_html(note(f"Alerts per hour: {failure_text(exc)}", "NA"))
    if hourly:
        bars = [(fmt_hour(r["hour"]), num(r["opened"]) or 0.0) for r in hourly]
        peak = max(bars, key=lambda b: b[1])
        emit_html(column_chart(bars, title="Alerts opened per hour", y_label="alerts",
                               desc=f"{len(bars)} hours, total {fmt_num(total['opened'])}, "
                                    f"peak {fmt_num(peak[1])} at {peak[0]}"))  # fmt: skip
    status_text = ""
    try:
        status = DATA.control("alerts_status")
        status_text = heading("Alert status now", 4) + html_table(status)
    except Exception as exc:
        status_text = note(f"Alert status: {failure_text(exc)}", "NA")
    emit_html(
        status_text
        + note(
            "Formulas: alerts = 'opened' events in the app's alert log within the window "
            "(one per alert episode). Mean time between alerts (MTBA) = (last opening - first "
            "opening) / (alerts - 1), which equals the mean gap between consecutive openings. "
            "Fault downtime = distinct minutes in which any of the asset's sensors reported "
            "fault_active. Operator stops = time the production cell was stopped from the app "
            "(production_event)."
        )
        + data_source_note(
            "Control-plane Lakebase (app alert engine) + UC Delta cold tier",
            ["alert, alert_event, production_event (app backend Lakebase)", PIPE.display()],
            WINDOW_TEXT,
            "Alerts are raised by the app's alert engine from the real-time feed and stored "
            "in its own Lakebase database; fault minutes come from the Delta telemetry.",
        )
    )


alarms_section()

## 2. Downtime and alarms

Alerts opened

288

in the 24 h window

Critical alerts

288

severity crit

Mean time between alerts

5.0 min

plant-wide

Fault downtime

576 min

sum of asset fault minutes (telemetry)

Operator stops

0s

production cell stopped from the app

table view

| KPI | value | detail |
|---|---|---|
| Alerts opened | 288 | in the 24 h window |
| Critical alerts | 288 | severity crit |
| Mean time between alerts | 5.0 min | plant-wide |
| Fault downtime | 576 min | sum of asset fault minutes (telemetry) |
| Operator stops | 0s | production cell stopped from the app |

#### Top offending assets and components

| asset | component | alerts | critical | MTBA | fault minutes (telemetry) |
|---|---|---|---|---|---|
| CNC-1 | Spindle 3 | 288 | 288 | 5.0 min | 576 |

#### By fault code

| fault code | rule | severity | alerts | first | last | MTBA |
|---|---|---|---|---|---|---|
| M-SPINDLE-OVERTEMP | mfg-alert-spindle-temp | crit | 288 | 2026-11-19 10:44 UTC | 2026-11-20 10:38 UTC | 5.0 min |

#### Alert status now

| status | severity | alerts |
|---|---|---|
| cleared | crit | 1 |
| cleared | warn | 1 |

Formulas: alerts = 'opened' events in the app's alert log within the window (one per alert episode). Mean time between alerts (MTBA) = (last opening - first opening) / (alerts - 1), which equals the mean gap between consecutive openings. Fault downtime = distinct minutes in which any of the asset's sensors reported fault_active. Operator stops = time the production cell was stopped from the app (production_event).

#### Data source

| item | value |
|---|---|
| tier | Control-plane Lakebase (app alert engine) + UC Delta cold tier |
| table | alert, alert_event, production_event (app backend Lakebase), <catalog>.<schema>.lb__shadow_manufacturing_sensor_readings_history |
| time window | last 24 h (2026-11-19 10:40 UTC to 2026-11-20 10:40 UTC) |
| note | Alerts are raised by the app's alert engine from the real-time feed and stored in its own Lakebase database; fault minutes come from the Delta telemetry. |

## 3. Sensor health trends

Key signals (spindle temperature, vibration, spindle load and press pressure) are tracked hour
by hour against their alert thresholds, and every sensor is checked for drift: has its newest
hour moved away from its usual level? The current value comes from the hot tier, the trend from
Delta.

In [ ]:
def _by_sensor(hourly: list[dict[str, Any]]) -> dict[str, list[dict[str, Any]]]:
    out: dict[str, list[dict[str, Any]]] = {}
    for r in sorted(hourly, key=lambda r: to_dt(r["hour"]) or WINDOW_START):
        out.setdefault(r["sensor_name"], []).append(r)
    return out


def sensor_health_rows(
    hourly: list[dict[str, Any]], live: dict[str, dict[str, Any]]
) -> list[dict[str, Any]]:
    """Per sensor: window mean, newest-hour mean, drift, threshold breaches, live value."""
    out = []
    for sensor, hours in _by_sensor(hourly).items():
        n = sum(num(h.get("n")) or 0 for h in hours)
        window_mean = weighted_mean((h.get("avg_value"), h.get("n")) for h in hours)
        recent = hours[-1]
        nominal = num(recent.get("nominal"))
        drift = drift_pct(recent.get("avg_value"), window_mean, nominal)
        rule = RULE_BY_SENSOR.get(sensor)
        breach_share = (
            safe_ratio(sum(num(h.get("breaches")) or 0 for h in hours), n) if rule else None
        )
        oor_share = safe_ratio(sum(num(h.get("out_of_range")) or 0 for h in hours), n)
        flags = []
        if drift is not None and abs(drift) >= DRIFT_PCT_OF_NOMINAL:
            flags.append("drifting")
        if breach_share is not None and 100 * breach_share >= BREACH_PCT_FLAG:
            flags.append("past alert threshold")
        if oor_share and 100 * oor_share >= BREACH_PCT_FLAG:
            flags.append("out of range")
        live_row = live.get(sensor) or {}
        out.append(
            {
                "sensor": sensor, "asset": asset_of(sensor),
                "window mean": fmt_num(window_mean, 2), "newest hour": fmt_num(recent.get("avg_value"), 2),
                "nominal": fmt_num(nominal, 1),
                "drift": "-" if drift is None else f"{drift:+.1f}%",
                "past threshold": fmt_pct(breach_share, 2) if rule else "no rule",
                "live now": fmt_num(live_row.get("avg_value"), 2) if live_row else "-",
                "status": ", ".join(flags) or "ok",
                "_drift": abs(drift or 0.0), "_flagged": bool(flags),
            }
        )  # fmt: skip
    out.sort(key=lambda r: (not r["_flagged"], -r["_drift"]))
    return out


def _trend_chart(sensor: str, label: str, unit: str, hours: list[dict[str, Any]]) -> str:
    rule = RULE_BY_SENSOR.get(sensor)
    if rule:
        word = "above" if rule.operator == "gt" else "below"
        threshold = (rule.threshold, f"alert {word} {fmt_num(rule.threshold)}")
    else:
        limit = num(hours[-1].get("range_hi")) if hours else None
        threshold = (limit, f"limit {fmt_num(limit)}") if limit is not None else None
    mean = [((to_dt(h["hour"]) or WINDOW_START).timestamp(), num(h["avg_value"]) or 0.0,
             fmt_hour(h["hour"])) for h in hours]  # fmt: skip
    peak_key = "min_seen" if rule and rule.operator == "lt" else "max_seen"
    peak_label = "hourly min" if peak_key == "min_seen" else "hourly max"
    peak = [((to_dt(h["hour"]) or WINDOW_START).timestamp(), num(h[peak_key]) or 0.0,
             fmt_hour(h["hour"])) for h in hours]  # fmt: skip
    desc = (
        f"{len(hours)} hours; mean {fmt_num(min(p[1] for p in mean), 2)} to "
        f"{fmt_num(max(p[1] for p in mean), 2)} {unit}; {peak_label} reached "
        f"{fmt_num((min if peak_key == 'min_seen' else max)(p[1] for p in peak), 2)} {unit}"
        + (f"; {threshold[1]}" if threshold else "")
    )
    return line_chart({"hourly mean": mean, peak_label: peak}, title=f"{label} ({unit})",
                      y_label=unit, desc=desc, threshold=threshold)  # fmt: skip


def sensor_section() -> None:
    blocks = [heading("3. Sensor health trends", 2)]
    try:
        hourly = DATA.cold("hourly")
    except Exception as exc:
        reason = PIPE.reason if not PIPE.catalog else failure_text(exc)
        emit_html(
            "".join(blocks)
            + not_available("Sensor health trends", f"the telemetry could not be read: {reason}.")
        )
        return
    live: dict[str, dict[str, Any]] = {}
    live_note = ""
    try:
        live = {r["sensor_name"]: r for r in DATA.hot("hot_live")}
    except Exception as exc:
        live_note = f"Live values (hot tier): {failure_text(exc)}"
    rows = sensor_health_rows(hourly, live)
    flagged = [r for r in rows if r["_flagged"]]
    SUMMARY.append({"KPI": "Sensors flagged (drift / threshold)",
                    "value": f"{len(flagged)} of {len(rows)}", "section": "3"})  # fmt: skip
    blocks.append(paragraph(
        f"{len(flagged)} of {len(rows)} sensors are flagged in this window"
        + (": " + ", ".join(f"{r['sensor']} ({r['status']})" for r in flagged[:6]) if flagged else ".")
    ))  # fmt: skip
    emit_html("".join(blocks))
    series = _by_sensor(hourly)
    for sensor, label, unit in KEY_SENSORS:
        if series.get(sensor):
            emit_html(_trend_chart(sensor, label, unit, series[sensor]))
        else:
            emit_html(note(f"{label}: no readings in the window", "NA"))
    table_rows = [
        {"sensor": s, "hour": fmt_hour(h["hour"]), "mean": fmt_num(h["avg_value"], 2),
         "min": fmt_num(h["min_seen"], 2), "max": fmt_num(h["max_seen"], 2),
         "past threshold": fmt_num(h.get("breaches")) if s in RULE_BY_SENSOR else "no rule"}
        for s, _, _ in KEY_SENSORS for h in series.get(s, [])[-6:]
    ]  # fmt: skip
    emit_html(
        "<details><summary>table view of the trend charts (newest 6 hours per sensor)</summary>"
        + html_table(table_rows, max_rows=60)
        + "</details>"
        + heading("Assets drifting out of range", 4)
        + html_table(
            rows,
            [
                "sensor",
                "asset",
                "window mean",
                "newest hour",
                "nominal",
                "drift",
                "past threshold",
                "live now",
                "status",
            ],
            25,
        )
        + (note(live_note, "NA") if live_note else "")
        + note(
            f"Formulas: drift = (mean of the newest hour - mean of the window) / nominal value; "
            f"flagged when |drift| >= {DRIFT_PCT_OF_NOMINAL:g}%. Past threshold = share of "
            f"readings beyond the sensor's alert rule (domain catalog: spindle temperature > 60 °C "
            f"crit, vibration > 8 mm/s warn, press pressure < 120 bar warn); flagged at >= "
            f"{BREACH_PCT_FLAG:g}%. Sensors without a rule are compared with their operating "
            "limit (max_value) on the charts. 'Live now' is the average over the newest "
            f"{HOT_LIVE_MINUTES} minutes read from the hot tier."
        )
        + data_source_note(
            "UC Delta cold tier (hourly trends) + Lakebase hot tier (live now)"
            if PIPE.has_hot_tier
            else "UC Delta table (hourly trends)",
            [PIPE.display()] + ([f"Lakebase {HOT_TABLE}"] if PIPE.has_hot_tier else []),
            WINDOW_TEXT + (f"; live values: {HOT_WINDOW_TEXT}" if PIPE.has_hot_tier else ""),
            "The hot tier answers 'what is it doing right now' with a short, bounded read; "
            "the multi-hour trend is aggregated on Delta.",
        )  # fmt: skip
    )


sensor_section()

## 3. Sensor health trends

1 of 19 sensors are flagged in this window: cnc_1_spindle_temp (past alert threshold)

sensor,hour,mean,min,max,past threshold
cnc_1_spindle_temp,11-20 05:00,50.22,37.47,75,"10,164"
cnc_1_spindle_temp,11-20 06:00,50.18,36.12,75,"10,210"
cnc_1_spindle_temp,11-20 07:00,50.18,36.39,75,"10,169"
cnc_1_spindle_temp,11-20 08:00,50.15,35.60,75,"10,186"
cnc_1_spindle_temp,11-20 09:00,50.08,36.46,75,"10,159"
cnc_1_spindle_temp,11-20 10:00,49.92,36.05,75,"6,738"
cnc_1_vibration,11-20 05:00,2.02,0.98,3.07,0
cnc_1_vibration,11-20 06:00,2.02,0.95,3.09,0
cnc_1_vibration,11-20 07:00,2.02,0.90,3.07,0
cnc_1_vibration,11-20 08:00,2.01,0.84,3.17,0


## 4. Throughput per hour versus target

Throughput is the rate each machine actually ran at, hour by hour, against its nominal (target)
rate. If the pipeline carried a production counter this section would also report units per
hour; it checks for one first and says so when the data does not support it.

In [ ]:
def throughput_rows(hourly: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Per hour and rate sensor: actual rate, target (nominal) rate, % of target, run minutes."""
    out = []
    by_sensor = _by_sensor(hourly)
    for asset, sensor, _unit, per_hour, label in THROUGHPUT_SENSORS:
        for h in by_sensor.get(sensor, []):
            avg, nominal = num(h.get("avg_value")), num(h.get("nominal"))
            observed, faults = num(h.get("observed_min")) or 0, num(h.get("fault_min")) or 0
            out.append(
                {
                    "asset": asset, "hour": fmt_hour(h["hour"]), "_hour": h["hour"], "measure": label,
                    "actual": None if avg is None else avg * per_hour,
                    "target": None if nominal is None else nominal * per_hour,
                    "of target": safe_ratio(avg, nominal),
                    "running minutes": max(0.0, observed - faults),
                }
            )  # fmt: skip
    return out


def throughput_section() -> None:
    blocks = [heading("4. Throughput per hour versus target", 2)]
    try:
        hourly = DATA.cold("hourly")
    except Exception as exc:
        reason = PIPE.reason if not PIPE.catalog else failure_text(exc)
        emit_html(
            "".join(blocks)
            + not_available("Throughput", f"the telemetry could not be read: {reason}.")
        )
        return
    counter_hours = _by_sensor(hourly).get(COUNTER_SENSOR, [])
    is_counter, produced = counter_production(counter_hours)
    if is_counter:
        blocks.append(stat_tiles([("Units produced", fmt_num(produced),
                                   f"{COUNTER_SENSOR} increase over the window")]))  # fmt: skip
    else:
        falling = sum(
            1 for h in counter_hours
            if (num(h.get("last_value")) or 0) < (num(h.get("first_value")) or 0)
        )  # fmt: skip
        evidence = (
            f"{COUNTER_SENSOR} is reported as a level that moves up and down around its nominal "
            f"value (it fell within {falling} of {len(counter_hours)} hours), not as an "
            "incrementing count"
            if counter_hours
            else f"no {COUNTER_SENSOR} readings arrived in the window"
        )
        blocks.append(not_available(
            "Production counts (units per hour)",
            f"{evidence}, and the plant model has no good-part or reject sensor. Throughput is "
            "shown instead as each machine's rate versus its nominal (target) rate.",
        ))  # fmt: skip
    rows = throughput_rows(hourly)
    if not rows:
        emit_html(
            "".join(blocks) + not_available("Throughput", "no rate-sensor readings in the window.")
        )
        return
    for asset, _sensor, _unit, _per_hour, label in THROUGHPUT_SENSORS:
        mine = [r for r in rows if r["asset"] == asset]
        if not mine:
            continue
        share = weighted_mean((r["of target"], 1) for r in mine)
        actual = weighted_mean((r["actual"], 1) for r in mine)
        target = weighted_mean((r["target"], 1) for r in mine)
        unit_word = label.split(" per ")[0]
        SUMMARY.append({"KPI": f"{asset} rate vs target", "value": fmt_pct(share), "section": "4"})
        blocks.append(paragraph(
            f"{asset} averaged {fmt_num(actual, 0)} {unit_word} per hour against a target of "
            f"{fmt_num(target, 0)} ({fmt_pct(share)} of target, mean over {len(mine)} clock "
            "hours; the first and last hour of the window are partial)."
        ))  # fmt: skip
    emit_html("".join(blocks))
    for asset, _sensor, _unit, _per_hour, label in THROUGHPUT_SENSORS:
        mine = [r for r in rows if r["asset"] == asset]
        if not mine:
            continue
        bars = [(r["hour"], r["actual"] or 0.0) for r in mine]
        target = mine[-1]["target"]
        low = min(mine, key=lambda r: r["of target"] or 0)
        emit_html(column_chart(
            bars, title=f"{asset}: {label}", y_label=label.split(" per ")[0],
            target=(target, f"target {fmt_num(target, 0)}") if target else None,
            desc=f"{len(bars)} hours; lowest {fmt_pct(low['of target'])} of target at {low['hour']}; "
                 f"target {fmt_num(target, 0)} per hour",
        ))  # fmt: skip
    table = [
        {"asset": r["asset"], "hour": r["hour"], "actual": fmt_num(r["actual"], 0),
         "target": fmt_num(r["target"], 0), "of target": fmt_pct(r["of target"]),
         "running minutes": fmt_num(r["running minutes"])}
        for r in rows
    ]  # fmt: skip
    emit_html(
        "<details><summary>table view (every hour)</summary>"
        + html_table(table, max_rows=400)
        + "</details>"
        + note(
            "Formulas: actual per hour = mean rate in the hour x 60 (conveyor m/min -> metres per "
            "hour; spindle RPM -> revolutions per hour); target = nominal_value x 60; of target "
            "= mean rate / nominal rate; running minutes = minutes with readings minus fault "
            "minutes. Hours where the cell was stopped from the app show fewer running minutes "
            "(the stopped machines send no readings)."
        )
        + data_source_note("UC Delta cold tier", [PIPE.display()], WINDOW_TEXT)
    )


throughput_section()

## 4. Throughput per hour versus target

Production counts (units per hour): not available from this pipeline
Because press_1_cycle_count is reported as a level that moves up and down around its nominal value (it fell within 14 of 25 hours), not as an incrementing count, and the plant model has no good-part or reject sensor. Throughput is shown instead as each machine's rate versus its nominal (target) rate.

Assembly line 1 averaged 485 conveyor metres per hour against a target of 480 (101.0% of target, mean over 25 clock hours; the first and last hour of the window are partial).

CNC-1 averaged 211,157 spindle revolutions per hour against a target of 210,000 (100.6% of target, mean over 25 clock hours; the first and last hour of the window are partial).

asset,hour,actual,target,of target,running minutes
Assembly line 1,11-19 10:00,485,480,101.1%,19
Assembly line 1,11-19 11:00,484,480,100.8%,60
Assembly line 1,11-19 12:00,485,480,101.0%,60
Assembly line 1,11-19 13:00,485,480,101.0%,60
Assembly line 1,11-19 14:00,485,480,101.0%,60
Assembly line 1,11-19 15:00,484,480,100.9%,60
Assembly line 1,11-19 16:00,485,480,101.0%,60
Assembly line 1,11-19 17:00,485,480,101.0%,60
Assembly line 1,11-19 18:00,484,480,100.8%,60
Assembly line 1,11-19 19:00,484,480,100.8%,60


## Summary

The headline numbers in one table, plus which data sources answered.

In [ ]:
def summary_section() -> None:
    sources = [{"data source": k, "state": v} for k, v in sorted(DATA.sources.items())]
    finished = datetime.now(UTC)
    emit_html(
        heading("Summary", 2)
        + sample_banner()
        + (
            html_table(SUMMARY, ["KPI", "value", "section"])
            if SUMMARY
            else not_deployed_block("KPI summary")
        )  # fmt: skip
        + heading("Data sources reached", 4)
        + (html_table(sources, ["data source", "state"]) if sources else paragraph("none"))
        + paragraph(
            f"Run finished {fmt_ts(finished)} in {fmt_seconds((finished - RUN_STARTED).total_seconds())}; "
            f"window {WINDOW_TEXT}; every query was read-only."
        )
    )
    if BACKEND is not None:
        BACKEND.close()


summary_section()

## Summary

| KPI | value | section |
|---|---|---|
| Readings analysed | 25,578,754 | glance |
| Plant OEE | 58.7% | 1 |
| Lowest-OEE asset | CNC-1 (58.6%) | 1 |
| Alerts opened | 288 | 2 |
| Mean time between alerts | 5.0 min | 2 |
| Top offending asset | CNC-1 (288 alerts) | 2 |
| Sensors flagged (drift / threshold) | 1 of 19 | 3 |
| Assembly line 1 rate vs target | 101.0% | 4 |
| CNC-1 rate vs target | 100.6% | 4 |

#### Data sources reached

| data source | state |
|---|---|
| Lakebase LakeTS (hot tier) | reached |
| UC Delta (cold tier) | reached |
| control-plane Lakebase (alerts) | reached |
| streaming metrics table | reached |

Run finished 2026-11-20 10:41 UTC in 67s; window last 24 h (2026-11-19 10:40 UTC to 2026-11-20 10:40 UTC); every query was read-only.